# Supervised Machine Learning 

In [ ]:
# Choose Intron for analysis and set random seed 
intron = 1
Intron = intron 
random_seed = 727

In [ ]:
# Load Libraries
import numpy as np
import pandas as pd
import xgboost as xgb
import sklearn
import os
from sklearn.model_selection import train_test_split 
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, make_scorer, average_precision_score, precision_recall_curve, PrecisionRecallDisplay
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import auc
import seaborn as sns
import shap
from sklearn.metrics import classification_report
import openpyxl
from itertools import cycle
import scipy
from scipy.stats import fisher_exact
from xgboost import plot_tree
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
# Create directory for saving images
directory_name = f'Intron {Intron} Figures'
if not os.path.exists(directory_name):
    os.makedirs(directory_name)

## Import edited SHAP package

In [ ]:
## Not needed if already done once 
# Upload the edited beeswarm and bar plot shap files (files available on github)
#new_bar = '/home/jupyter/workspaces/conservationandgeneticvariabilityofintronsincftr/_bar.py'
#new_beeswarm = '/home/jupyter/workspaces/conservationandgeneticvariabilityofintronsincftr/_beeswarm.py'

#beeswarm_path = "/home/jupyter/.local/lib/python3.10/site-packages/shap/plots/_beeswarm.py"
#bar_path = "/home/jupyter/.local/lib/python3.10/site-packages/shap/plots/_bar.py"

#with open(new_beeswarm, 'r') as file:
#    new_content = file.read()

#with open(beeswarm_path, 'w') as file:
#    file.write(new_content)

#with open(new_bar, 'r') as file:
#    new_content = file.read()

#with open(bar_path, 'w') as file:
#    file.write(new_content)


In [ ]:
workspace_bucket = os.environ['WORKSPACE_BUCKET']

## Combined Model 

### Set Intron,load data for that Intron and engineer classification for F508del and V470M mutations in conjunction

In [ ]:
temp_data = pd.read_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/adjusted/Intron_{intron}.xlsx')
temp_data = temp_data[pd.notna(temp_data['V470M'])]
temp_data=temp_data.reset_index(drop=True)
temp_data1 = temp_data
temp_data1['F508del & V470M Combination'] = 0
temp_data1

for x in range(temp_data.shape[0]):
    if ((temp_data.loc[x, 'V470M'] == 0) & (temp_data.loc[x, 'F508del'] == 0)):
        temp_data1.loc[x, 'F508del & V470M Combination'] = 0
    elif ((temp_data.loc[x, 'V470M'] == 1) & (temp_data.loc[x, 'F508del'] == 0)):
        temp_data1.loc[x, 'F508del & V470M Combination'] = 1
    elif((temp_data.loc[x, 'V470M']== 2) & (temp_data.loc[x, 'F508del'] == 0)):
        temp_data1.loc[x, 'F508del & V470M Combination'] = 2
    elif ((temp_data.loc[x, 'V470M']== 2) & (temp_data.loc[x, 'F508del'] == 1)):
        temp_data1.loc[x, 'F508del & V470M Combination'] = 3
    elif ((temp_data.loc[x, 'V470M']== 2) & (temp_data.loc[x, 'F508del'] == 2)):
        temp_data1.loc[x, 'F508del & V470M Combination'] = 4
    elif ((temp_data.loc[x, 'V470M']== 1) & (temp_data.loc[x, 'F508del'] == 1)):
        temp_data1.loc[x, 'F508del & V470M Combination'] = 5

### Split Target Feature from Predictor Features 

In [ ]:
# Set dependent values for ML F508del + V470M
y = temp_data1['F508del & V470M Combination']

# Set independent Values
X = temp_data1.drop(columns=['F508del', 'V470M', 'F508del & V470M Combination'])

### Assess Correlations between Predictor Features 

In [ ]:
def correlation_heatmap(train):
    correlations = train.corr(method='spearman')

    fig, ax = plt.subplots(figsize=(20,20))
    mask = np.zeros_like(correlations, dtype=bool)
    mask[np.triu_indices_from(mask)] = True

    cmap1 = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(correlations, vmax=1.0, center = 0, fmt = '.2f', cmap = sns.diverging_palette(230, 20, as_cmap=True), mask = mask, square = True, linewidths = 0.5, annot=True, cbar_kws={"shrink": .70}, annot_kws={'fontweight': 'bold'})

    for i in range(correlations.shape[0]):
        ax.text(i+0.5, i+0.5, 'X', ha='center', va='center', color='red')
    ax.set_xticklabels(ax.get_xticklabels(), fontsize='large', fontweight='bold')
    ax.set_yticklabels(ax.get_yticklabels(), fontsize='large', fontweight='bold')
    plt.tight_layout()
    plt.show()

correlation_heatmap(X)

### Split data into Test and Training sets

In [ ]:
# Split into independent (X) training and testing data, and dependent (Y) training and testing data
X_train1, X_test1, y_train1, y_test1 = train_test_split(X, y,  random_state = random_seed, stratify = y)

# Get counts for each class 
cat_0 = len(y_test1[y_test1==0])
cat_1 = len(y_test1[y_test1==1])
cat_2 = len(y_test1[y_test1==2])
cat_3 = len(y_test1[y_test1==3])
cat_4 = len(y_test1[y_test1==4])
cat_5 = len(y_test1[y_test1==5])

# Confirm if stratification occured (Should be True)
print((abs(len(y_train1[y_train1 == 0])/len(y_train1))-(len(y_test1[y_test1 == 0])/len(y_test1))) <= 0.1)
print((abs(len(y_train1[y_train1 == 1])/len(y_train1))-(len(y_test1[y_test1 == 1])/len(y_test1))) <= 0.1)
print((abs(len(y_train1[y_train1 == 2])/len(y_train1))-(len(y_test1[y_test1 == 2])/len(y_test1))) <= 0.1)
print((abs(len(y_train1[y_train1 == 3])/len(y_train1))-(len(y_test1[y_test1 == 3])/len(y_test1))) <= 0.1)
print((abs(len(y_train1[y_train1 == 4])/len(y_train1))-(len(y_test1[y_test1 == 4])/len(y_test1))) <= 0.1)
print((abs(len(y_train1[y_train1 == 5])/len(y_train1))-(len(y_test1[y_test1 == 5])/len(y_test1))) <= 0.1)

# Check if there are any missing values in the arrays (missing values are fine) 
print(np.any(np.isnan(X_train1)) or np.any(np.isnan(y_train1)) or np.any(np.isnan(X_test1)) or np.any(np.isnan(y_test1)))

# Make sure locus match 
alleles = pd.read_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/adjusted/Intron_{intron}_Alleles.xlsx')
(alleles['locus'] == X.columns).all()

### Access Master file to retrieve Optimal Hyperparameters derived from running GridSearchCV

In [ ]:
mutation_check = 'V470M + F508del'

Hyperparameter_master_file = pd.read_excel(f'{workspace_bucket}/data/XGBoost ML Hyperparameters | Master File.xlsx')

index = Hyperparameter_master_file.loc[(Hyperparameter_master_file['Intron'] == Intron) & (Hyperparameter_master_file['Model'] == mutation_check)].index
ae= int(Hyperparameter_master_file.loc[index, 'n_estimators'])
be= int(Hyperparameter_master_file.loc[index, 'max_depth'])
ce= float(Hyperparameter_master_file.loc[index, 'learning_rate'])
de= float(Hyperparameter_master_file.loc[index, 'gamma'])
ee= float(Hyperparameter_master_file.loc[index, 'reg_lambda'])

### Train Model and predict results using the testing dataset

In [ ]:
# fitting for the first dataset (nucleotides as the independent variable), test with the parameters determined in the previous steps 
clf_xgb1 = xgb.XGBClassifier(objective='multi:softprob', eval_metric='aucpr', seed=random_seed, n_estimators = ae
                             , max_depth = be , learning_rate = ce , gamma= de, 
                             reg_lambda = ee) # note, objective set to binary:logistic as this dataset is being used for classification 
# Stopping the tree building early if the evaluation metrics (in this case AUCPR) decreases for 10 rounds in a row 
clf_xgb1.set_params(early_stopping_rounds= 10)
clf_xgb1.fit(X_train1, y_train1, verbose = True, eval_set=[(X_test1, y_test1)])
y_pred1 = clf_xgb1.predict(X_test1)

In [ ]:
# Determine the best round and the aucpr score associated with it 
print('Best Round:', clf_xgb1.best_iteration)
print('AUCPR Score for the Best Round:', clf_xgb1.best_score)

### Interpreting the Model: Mean SHAP Value (% Contribution to the Model)

In [ ]:
# Calculating the Mean Shapley Value for each SNP to each respective associated CFTR Genetic Variant, then calculating what percentage of the CFTR Genetic Variant's prediction is described by the SNP

Mean_abs_shap_df = pd.DataFrame(columns=['Intron', 'CFTR Genetic Variant','SNP Locus', 'Mean Absolute Shapley Value', '% Contribution of SNP towards CFTR Genetic Variant Prediction'])
Index = 0
shap_values = shap.Explainer(clf_xgb1).shap_values(X_test1)

for x in range(6): # range of 6 because of 3 F508del+V470M classes
    values = abs(shap_values).mean(0)[:,x] # Obtain shap values for all individuals, depending on class (x)
    sorted_index = np.argsort(values)[::-1] # Retrieve Index based on descending order of the previous values calculated

    for y in range((values> 0).sum()): # > not really necessary as the absolute value of all shapley values is taken 
        Mean_abs_shap_df.loc[Index, 'Intron'] = Intron
        Mean_abs_shap_df.loc[Index, 'CFTR Genetic Variant'] = x
        Mean_abs_shap_df.loc[Index, 'SNP Locus'] = X_train1.columns[sorted_index][y]
        Mean_abs_shap_df.loc[Index, 'Mean Absolute Shapley Value'] = values[sorted_index][y]
        Index += 1 # Incremental increases to Index


# Calculate total amount of SHAP values towards predicting each respective class
sum_0 = np.sum(Mean_abs_shap_df[Mean_abs_shap_df['CFTR Genetic Variant'] == 0]['Mean Absolute Shapley Value'])
sum_1 = np.sum(Mean_abs_shap_df[Mean_abs_shap_df['CFTR Genetic Variant'] == 1]['Mean Absolute Shapley Value'])
sum_2 = np.sum(Mean_abs_shap_df[Mean_abs_shap_df['CFTR Genetic Variant'] == 2]['Mean Absolute Shapley Value'])
sum_3 = np.sum(Mean_abs_shap_df[Mean_abs_shap_df['CFTR Genetic Variant'] == 3]['Mean Absolute Shapley Value'])
sum_4 = np.sum(Mean_abs_shap_df[Mean_abs_shap_df['CFTR Genetic Variant'] == 4]['Mean Absolute Shapley Value'])
sum_5 = np.sum(Mean_abs_shap_df[Mean_abs_shap_df['CFTR Genetic Variant'] == 5]['Mean Absolute Shapley Value'])

# Calculate percentage contribution of each SNP towards the class 
for x in range(len(Mean_abs_shap_df)):
    if Mean_abs_shap_df['CFTR Genetic Variant'][x] == 0:
        Mean_abs_shap_df.loc[x,'% Contribution of SNP towards CFTR Genetic Variant Prediction'] = ((Mean_abs_shap_df['Mean Absolute Shapley Value'][x])/sum_0)*100
    elif Mean_abs_shap_df['CFTR Genetic Variant'][x] == 1:
        Mean_abs_shap_df.loc[x,'% Contribution of SNP towards CFTR Genetic Variant Prediction'] = ((Mean_abs_shap_df['Mean Absolute Shapley Value'][x])/sum_1)*100
    elif Mean_abs_shap_df['CFTR Genetic Variant'][x] == 2:
        Mean_abs_shap_df.loc[x,'% Contribution of SNP towards CFTR Genetic Variant Prediction'] = ((Mean_abs_shap_df['Mean Absolute Shapley Value'][x])/sum_2)*100
    elif Mean_abs_shap_df['CFTR Genetic Variant'][x] == 3:
        Mean_abs_shap_df.loc[x,'% Contribution of SNP towards CFTR Genetic Variant Prediction'] = ((Mean_abs_shap_df['Mean Absolute Shapley Value'][x])/sum_3)*100
    elif Mean_abs_shap_df['CFTR Genetic Variant'][x] == 4:
        Mean_abs_shap_df.loc[x,'% Contribution of SNP towards CFTR Genetic Variant Prediction'] = ((Mean_abs_shap_df['Mean Absolute Shapley Value'][x])/sum_4)*100
    elif Mean_abs_shap_df['CFTR Genetic Variant'][x] == 5:
        Mean_abs_shap_df.loc[x,'% Contribution of SNP towards CFTR Genetic Variant Prediction'] = ((Mean_abs_shap_df['Mean Absolute Shapley Value'][x])/sum_5)*100

Mean_abs_shap_df['CFTR Genetic Variant'] = Mean_abs_shap_df['CFTR Genetic Variant'].replace([0,1,2,3,4,5], ["V/V & No F508del Mutation", "V/M & No F508del Mutation", "M/M & No F508del Mutation", 'M/M & Heterozygous F508del Mutation',
                                                                        'M/M & Homozygous F508del Mutation', 'V/M & Heterozygous F508del Mutation'])
Mean_abs_shap_df.to_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {Intron} | Mean Absolute Shapley Values | V470M+F508del.xlsx', index=False)
Mean_abs_shap_df

In [ ]:
ax = sns.barplot(Mean_abs_shap_df, errorbar=("pi"), capsize=.1, x='SNP Locus', y='% Contribution of SNP towards CFTR Genetic Variant Prediction', edgecolor='green', facecolor='white')
x_values = [p.get_text() for p in ax.get_xticklabels()]
y_values = [p.get_height() for p in ax.patches]
allele_values = []

for x in range(len(x_values)):
    for y in range(len(alleles)):
                   if x_values[x] == alleles['locus'][y]:
                    allele_values.append(alleles['alleles'][y])
        
result_df = pd.DataFrame({'SNP Locus': x_values, 'Alleles' : allele_values, 'Mean Contribution': y_values})
result_df.sort_values('Mean Contribution', ascending=False, inplace=True)
result_df.to_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {Intron} | Mean contribution of SNP towards prediction of associated CFTR Genetic Variants | V470M+F508del.xlsx', index=False)
result_df

In [ ]:
# Creating a figure which visualizes what visualizes the contribution of each SNP towards prediction in each respective associated class (also shows the mean impact)

fig = plt.figure(figsize=(9,9))
Mean_abs_shap_df.sort_values('% Contribution of SNP towards CFTR Genetic Variant Prediction', ascending = False, inplace=True)
sns.barplot(Mean_abs_shap_df, errorbar='se', capsize=.1, x='SNP Locus', y='% Contribution of SNP towards CFTR Genetic Variant Prediction', edgecolor='black', facecolor='white', order=result_df['SNP Locus'], errcolor = 'silver')
plt.yticks(fontsize='x-large')
plt.xticks(rotation=45, ha='right', fontsize='x-large')
plt.xlabel('SNP Locus', fontsize='x-large', fontweight='bold')
plt.ylabel('% Contribution of SNP towards CFTR Genetic \n Variant Prediction', fontsize='x-large', fontweight='bold')
sns.swarmplot(Mean_abs_shap_df, x='SNP Locus', y= '% Contribution of SNP towards CFTR Genetic Variant Prediction', hue='CFTR Genetic Variant', hue_order=["V/V & No F508del Mutation", "V/M & No F508del Mutation", "M/M & No F508del Mutation", 'M/M & Heterozygous F508del Mutation',
                                                                        'M/M & Homozygous F508del Mutation', 'V/M & Heterozygous F508del Mutation'], palette=['skyblue', 'orange', 'green', 'red', 'violet', 'blue'], edgecolor= 'black', linewidth= 0.5,size = 7.5)
plt.legend(title='CFTR Genetic Variant', title_fontsize='x-large', fontsize='x-large')
plt.tight_layout()
filename = f'Intron {Intron} | V470M+F508del | SNP Contribution.pdf'
file_path = os.path.join(directory_name, filename)
plt.savefig(file_path, format='pdf', dpi=600, bbox_inches='tight')
plt.show()

### Determine important SNPs which may be masked due to high correlation

In [ ]:
# Determine Correlation of SNPs deemed important by the model, with other intronic SNPs
correlations = X.corr(method='spearman')
temp_filtered_SNPs = result_df.set_index('SNP Locus')
temp_full_SNPs = alleles.set_index('locus')
correlations_file = pd.DataFrame(columns=('Intron', 'Main SNP Locus', 'Alleles of Main SNP', 'Mean Contribution of main SNP', 'Associated SNP Locus', 'Alleles of Associated SNP', 'Meets Correlation Threshold of 0.9?', 'Correlation'))
index = 0
for x in range(len(x_values)):
    for y in range(len(correlations)):
        if ((correlations[x_values[x]][y] >= 0.7) & (x_values[x] != correlations.index[y])):
            correlations_file.at[index, 'Intron'] = Intron
            correlations_file.at[index, 'Main SNP Locus'] = x_values[x]
            correlations_file.at[index, 'Alleles of Main SNP'] = temp_filtered_SNPs.at[x_values[x], 'Alleles']
            correlations_file.at[index, 'Mean Contribution of main SNP'] = temp_filtered_SNPs.at[x_values[x],'Mean Contribution']
            correlations_file.at[index, 'Associated SNP Locus'] = correlations.index[y]
            correlations_file.at[index, 'Alleles of Associated SNP'] = temp_full_SNPs.at[correlations.index[y], 'alleles']
            correlations_file.at[index, 'Meets Correlation Threshold of 0.9?'] = 'Yes' if correlations[x_values[x]][y] >= 0.9 else 'No'
            correlations_file.at[index, 'Correlation'] = correlations[x_values[x]][y]
            index += 1

correlations_file.to_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {Intron} | SNP Correlations File (filtered for 0.7 threshold) | V470M+F508del.xlsx', index='False')   
correlations_file

In [ ]:
# Compile possible SNPS that pass thresholds of 0.7 and 0.9 respectively, for correlation with SNPs important to the model (making sure not to list correlations with other SNPs deemed important by the model)
Final_SNP_Selection = correlations_file[correlations_file['Associated SNP Locus'].isin(result_df['SNP Locus']) == False]

# For 0.7 Threshold
Final_SNP_Selection['Mean Contribution of main SNP'] = pd.to_numeric(
    Final_SNP_Selection['Mean Contribution of main SNP'], errors='coerce')
max_threshold_rows = Final_SNP_Selection.groupby('Associated SNP Locus')['Mean Contribution of main SNP'].idxmax()
Final_SNP_Selection = Final_SNP_Selection.loc[max_threshold_rows]
Final_SNP_Selection = Final_SNP_Selection.reset_index(drop=True)


# For 0.9 Threshold
Final_SNP_Selection2 = Final_SNP_Selection[Final_SNP_Selection['Meets Correlation Threshold of 0.9?'] == 'Yes']
max_threshold_rows2 = Final_SNP_Selection2.groupby('Associated SNP Locus')['Mean Contribution of main SNP'].idxmax()
Final_SNP_Selection2 = Final_SNP_Selection2.loc[max_threshold_rows2]
Final_SNP_Selection2 = Final_SNP_Selection2.reset_index(drop=True)

Final_SNP_Selection2

In [ ]:
# Final SNP table with 0.7 Threshold
Final_SNP_0_7 = result_df
Final_SNP_0_7['Mean Contribution of Main SNP'] = Final_SNP_0_7['Mean Contribution']
Final_SNP_0_7 = Final_SNP_0_7.drop(columns='Mean Contribution')
Final_SNP_0_7['Associated SNP Locus'] = ''
Final_SNP_0_7['Correlation'] = ''
index1 = len(Final_SNP_0_7)

for x in range(len(Final_SNP_Selection)):
    Final_SNP_0_7.at[index1,'SNP Locus'] = Final_SNP_Selection['Associated SNP Locus'][x]
    Final_SNP_0_7.at[index1,'Associated SNP Locus'] = Final_SNP_Selection['Main SNP Locus'][x]
    Final_SNP_0_7.at[index1,'Alleles'] = Final_SNP_Selection['Alleles of Associated SNP'][x]
    Final_SNP_0_7.at[index1,'Mean Contribution of Main SNP'] = Final_SNP_Selection['Mean Contribution of main SNP'][x]
    Final_SNP_0_7.at[index1,'Correlation'] = Final_SNP_Selection['Correlation'][x]
    index1 += 1

Final_SNP_0_7 = Final_SNP_0_7.set_index('SNP Locus')

Final_SNP_0_9 = result_df
Final_SNP_0_9['Mean Contribution of Main SNP'] = Final_SNP_0_9['Mean Contribution']
Final_SNP_0_9 = Final_SNP_0_9.drop(columns='Mean Contribution')
Final_SNP_0_9['Associated SNP Locus'] = ''
Final_SNP_0_9['Correlation'] = ''
index2 = len(Final_SNP_0_9)

# Final SNP table with 0.9 Threshold 
for x in range(len(Final_SNP_Selection2)):
    Final_SNP_0_9.at[index2,'SNP Locus'] = Final_SNP_Selection2['Associated SNP Locus'][x]
    Final_SNP_0_9.at[index2,'Associated SNP Locus'] = Final_SNP_Selection2['Main SNP Locus'][x]
    Final_SNP_0_9.at[index2,'Alleles'] = Final_SNP_Selection2['Alleles of Associated SNP'][x]
    Final_SNP_0_9.at[index2,'Mean Contribution of Main SNP'] = Final_SNP_Selection2['Mean Contribution of main SNP'][x]
    Final_SNP_0_9.at[index2,'Correlation'] = Final_SNP_Selection2['Correlation'][x]
    index2 += 1

Final_SNP_0_9 = Final_SNP_0_9.set_index('SNP Locus')
Final_SNP_0_9

### Interpreting the Model: Mean SHAP Value (contribution to each class)

In [ ]:
shap_values = shap.Explainer(clf_xgb1).shap_values(X_test1)
max_0 = np.sum(abs(shap_values).mean(0)[:,0] > 0 ) + 1
max_1 = np.sum(abs(shap_values).mean(0)[:,1] > 0 ) + 1 
max_2 = np.sum(abs(shap_values).mean(0)[:,2] > 0 ) + 1
max_3 = np.sum(abs(shap_values).mean(0)[:,3] > 0 ) + 1
max_4 = np.sum(abs(shap_values).mean(0)[:,4] > 0 ) + 1
max_5 = np.sum(abs(shap_values).mean(0)[:,5] > 0 ) + 1

labels = ["V/V & No F508del Mutation", "V/M & No F508del Mutation", "M/M & No F508del Mutation", 'M/M & Heterozygous F508del Mutation',
                                                                        'M/M & Homozygous F508del Mutation', 'V/M & Heterozygous F508del Mutation']
explainer = shap.Explainer(clf_xgb1)
shap_values = explainer(X_test1)

maxy = max(max_0,max_1,max_2, max_3, max_4, max_5) # obtain max number of SNPs amongst each class
max_x_abs = np.max(np.average(np.abs(shap_values.values), axis=(0))) # Obtain SNP importance of the  to set it as x axis limit

plt.subplot(2,3,1)
shap.plots.bar(shap_values[:,:,0], max_display= maxy, show=False)
plt.title(labels[0] + f' (n={cat_0})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='xx-large')
plt.yticks(fontsize='xx-large', fontweight='bold')
plt.xlabel('mean (|SHAP value|)', fontsize='xx-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(0, max_x_abs)

plt.subplot(2,3,2)
shap.plots.bar(shap_values[:,:,1], max_display= maxy, show=False)
plt.title(labels[1] +  f' (n={cat_1})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='xx-large')
plt.yticks(fontsize='xx-large', fontweight='bold')
plt.xlabel('mean (|SHAP value|)', fontsize='xx-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(0, max_x_abs)

plt.subplot(2,3,3)
shap.plots.bar(shap_values[:,:,2], max_display= maxy, show=False)
plt.title(labels[2] + f' (n={cat_2})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='xx-large')
plt.yticks(fontsize='xx-large', fontweight='bold')
plt.xlabel('mean (|SHAP value|)', fontsize='xx-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(0, max_x_abs)

plt.subplot(2,3,4)
shap.plots.bar(shap_values[:,:,3], max_display= maxy, show=False)
plt.title(labels[3] +  f' (n={cat_3})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='xx-large')
plt.yticks(fontsize='xx-large', fontweight='bold')
plt.xlabel('mean (|SHAP value|)', fontsize='xx-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(0, max_x_abs)

plt.subplot(2,3,5)
shap.plots.bar(shap_values[:,:,4], max_display= maxy, show=False)
plt.title(labels[4] +  f' (n={cat_4})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='xx-large')
plt.yticks(fontsize='xx-large', fontweight='bold')
plt.xlabel('mean (|SHAP value|)', fontsize='x-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(0, max_x_abs)

plt.subplot(2,3,6)
shap.plots.bar(shap_values[:,:,5], max_display= maxy, show=False)
plt.title(labels[5] +  f' (n={cat_5})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='xx-large')
plt.yticks(fontsize='xx-large', fontweight='bold')
plt.xlabel('mean (|SHAP value|)', fontsize='xx-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(0, max_x_abs)

plt.subplots_adjust(top=3, bottom=0.05, left=0.05, right=3.5, hspace=0.3, wspace=0.55)
filename = f'Intron {Intron} | V470M+F508del | Barplot.pdf'
file_path = os.path.join(directory_name, filename)
plt.savefig(file_path, format='pdf', dpi=600, bbox_inches='tight')
plt.show()


### Interpreting the Model: SHAP Value (directional contribution to each class)

In [ ]:
min_x = np.min(shap_values.values) - (abs(np.min(shap_values.values))*0.01)  # obtain minimum shap value to set x-axis limit, giving slight buffer space aswell
max_x = np.max(shap_values.values) + (abs(np.max(shap_values.values))*0.01) # obtain maximum shap value to set x-axis limit, giving slight buffer space aswell 

plt.subplot(2,3,1)
shap.plots.beeswarm(shap_values[:,:,0], max_display= maxy, color_bar_label='Genotype Call Value of SNP',show=False)
cbar = plt.gcf().axes[-1]
vmin, vmax = cbar.get_ylim()
new_ticks = [vmin, (vmin+vmax)/2, vmax]
new_labels = ['0/0', '0/1\nor\n1/0', '1/1']
cbar.set_yticks(new_ticks)
cbar.set_yticklabels(new_labels)
cbar.yaxis.set_label_coords(9.5, 0.5)
plt.title(labels[0] + f' (n={cat_0})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='xx-large')
plt.yticks(fontsize='xx-large', fontweight='bold')
plt.xlabel('SHAP value', fontsize='xx-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(min_x, max_x)

plt.subplot(2,3,2)
shap.plots.beeswarm(shap_values[:,:,1], max_display= maxy, color_bar_label='Genotype Call Value of SNP',show=False)
cbar = plt.gcf().axes[-1]
vmin, vmax = cbar.get_ylim()
new_ticks = [vmin, (vmin+vmax)/2, vmax]
new_labels = ['0/0', '0/1\nor\n1/0', '1/1']
cbar.set_yticks(new_ticks)
cbar.set_yticklabels(new_labels)
cbar.yaxis.set_label_coords(9.5, 0.5)
plt.title(labels[1] + f' (n={cat_1})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='xx-large')
plt.yticks(fontsize='xx-large', fontweight='bold')
plt.xlabel('SHAP value', fontsize='xx-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(min_x, max_x)

plt.subplot(2,3,3)
shap.plots.beeswarm(shap_values[:,:,2], max_display= maxy, color_bar_label='Genotype Call Value of SNP',show=False)
cbar = plt.gcf().axes[-1]
vmin, vmax = cbar.get_ylim()
new_ticks = [vmin, (vmin+vmax)/2, vmax]
new_labels = ['0/0', '0/1\nor\n1/0', '1/1']
cbar.set_yticks(new_ticks)
cbar.set_yticklabels(new_labels)
cbar.yaxis.set_label_coords(9.5, 0.5)
plt.title(labels[2] + f' (n={cat_2})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='xx-large')
plt.yticks(fontsize='xx-large', fontweight='bold')
plt.xlabel('SHAP value', fontsize='xx-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(min_x, max_x)

plt.subplot(2,3,4)
shap.plots.beeswarm(shap_values[:,:,3], max_display= maxy, color_bar_label='Genotype Call Value of SNP', show=False)
cbar = plt.gcf().axes[-1]
vmin, vmax = cbar.get_ylim()
new_ticks = [vmin, (vmin+vmax)/2, vmax]
new_labels = ['0/0', '0/1\nor\n1/0', '1/1']
cbar.set_yticks(new_ticks)
cbar.set_yticklabels(new_labels)
cbar.yaxis.set_label_coords(9.5, 0.5)
plt.title(labels[3] + f' (n={cat_3})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='xx-large')
plt.yticks(fontsize='xx-large', fontweight='bold')
plt.xlabel('SHAP value', fontsize='xx-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(min_x, max_x)

plt.subplot(2,3,5)
shap.plots.beeswarm(shap_values[:,:,4], max_display= maxy, color_bar_label='Genotype Call Value of SNP',show=False)
cbar = plt.gcf().axes[-1]
vmin, vmax = cbar.get_ylim()
new_ticks = [vmin, (vmin+vmax)/2, vmax]
new_labels = ['0/0', '0/1\nor\n1/0', '1/1']
cbar.set_yticks(new_ticks)
cbar.set_yticklabels(new_labels)
cbar.yaxis.set_label_coords(9.5, 0.5)
plt.title(labels[4] +  f' (n={cat_4})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='xx-large')
plt.yticks(fontsize='xx-large', fontweight='bold')
plt.xlabel('SHAP value', fontsize='xx-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(min_x, max_x)

plt.subplot(2,3,6)
shap.plots.beeswarm(shap_values[:,:,5], max_display= maxy, color_bar_label='Genotype Call Value of SNP',show=False)
cbar = plt.gcf().axes[-1]
vmin, vmax = cbar.get_ylim()
new_ticks = [vmin, (vmin+vmax)/2, vmax]
new_labels = ['0/0', '0/1\nor\n1/0', '1/1']
cbar.set_yticks(new_ticks)
cbar.set_yticklabels(new_labels)
cbar.yaxis.set_label_coords(9.5, 0.5)
plt.title(labels[5] +  f' (n={cat_5})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='xx-large')
plt.yticks(fontsize='xx-large', fontweight='bold')
plt.xlabel('SHAP value', fontsize='xx-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(min_x, max_x)

plt.subplots_adjust(top=3, bottom=0.05, left=0.05, right=3.5, hspace=0.3, wspace=0.55)
filename = f'Intron {Intron} | V470M+F508del | Dotplot.pdf'
file_path = os.path.join(directory_name, filename)
plt.savefig(file_path, format='pdf', dpi=100, bbox_inches='tight')
plt.show()

### Assessing Model Performance: Confusion Matrix 

In [ ]:
# Building a confusion matrix 
cm1 = confusion_matrix(y_test1, y_pred1)
cm1_percent = cm1 / (cm1.sum(axis=1)[:, np.newaxis])

disp1 = ConfusionMatrixDisplay(confusion_matrix = cm1, display_labels= ["V/V & No F508del Mutation", "V/M & No F508del Mutation", "M/M & No F508del Mutation", 'M/M & Heterozygous F508del Mutation',
                                                                        'M/M & Homozygous F508del Mutation', 'V/M & Heterozygous F508del Mutation'])
disp2 = ConfusionMatrixDisplay(confusion_matrix = cm1_percent, display_labels= ["V/V & No F508del Mutation", "V/M & No F508del Mutation", "M/M & No F508del Mutation", 'M/M & Heterozygous F508del Mutation',
                                                                        'M/M & Homozygous F508del Mutation', 'V/M & Heterozygous F508del Mutation'])
fig, ax = plt.subplots(1,2, figsize=(24,12))
disp1.plot(ax=ax[0])
disp2.plot(ax=ax[1], values_format='.1%')
disp1.ax_.set_xticklabels(disp1.ax_.get_xticklabels(), rotation=45, ha='right', fontsize='x-large')
disp1.ax_.set_yticklabels(disp1.ax_.get_yticklabels(), fontsize='x-large')
disp1.ax_.set_ylabel('True Label', fontsize='x-large', fontweight='bold')
disp1.ax_.set_xlabel('Predicted Label', fontsize='x-large', fontweight='bold')
disp2.ax_.set_xticklabels(disp2.ax_.get_xticklabels(), rotation=45, ha='right', fontsize='x-large')
disp2.ax_.set_yticklabels(disp2.ax_.get_yticklabels(), fontsize='x-large')
disp2.ax_.set_ylabel('True Label', fontsize='x-large', fontweight='bold')
disp2.ax_.set_xlabel('Predicted Label', fontsize='x-large', fontweight='bold')
disp1.ax_.xaxis.labelpad = 13
disp2.ax_.xaxis.labelpad = 13
disp1.ax_.yaxis.labelpad = 13
disp2.ax_.yaxis.labelpad = 13
# Change font size within the confusion matrix
for axis in ax:
    for text in axis.texts:
        text.set_fontsize('xx-large')
# Add red borders diagonally 
for i in range(len(cm1)):
    disp1.ax_.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1, fill=False, edgecolor='red', lw=3, zorder = 10))
for i in range(len(cm1_percent)):
    disp2.ax_.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1, fill=False, edgecolor='red', lw=3, zorder = 10))
plt.subplots_adjust(wspace=0.65)
filename = f'Intron {Intron} | V470M+F508del | Confusion Matrix.pdf'
file_path = os.path.join(directory_name, filename)
#fig.suptitle(f'Confusion Matrix of {mutation} Mutation Classification for Intron {Intron} ', y = 0.92)
plt.savefig(file_path, format='pdf', dpi=600, bbox_inches='tight')
fig.show()


### Assessing Model Performance: Precision-Recall Curves

In [ ]:
# Convert y_test1 to the same format as y_pred1 (rather than having an array of shape [77,1] have an array with shape [77,3] with each column depicting which class the individual is)
y_pred1 = clf_xgb1.predict_proba(X_test1)
y_pred1_multiclass = clf_xgb1.predict_proba(X_test1)
y_test1_temp = y_test1.reset_index(drop=True).copy()
y_test1_multiclass = np.zeros((y_pred1.shape[0],y_pred1.shape[1]))

for x in range(y_test1_temp.shape[0]):
    if y_test1_temp[x] == 0:
        y_test1_multiclass[x,0] = 1
        y_test1_multiclass[x,1] = 0
        y_test1_multiclass[x,2] = 0
        y_test1_multiclass[x,3] = 0
        y_test1_multiclass[x,4] = 0
        y_test1_multiclass[x,5] = 0


    elif y_test1_temp[x] == 1:
        y_test1_multiclass[x,0] = 0
        y_test1_multiclass[x,1] = 1
        y_test1_multiclass[x,2] = 0
        y_test1_multiclass[x,3] = 0
        y_test1_multiclass[x,4] = 0
        y_test1_multiclass[x,5] = 0


    elif y_test1_temp[x] == 2:
        y_test1_multiclass[x,0] = 0
        y_test1_multiclass[x,1] = 0
        y_test1_multiclass[x,2] = 1
        y_test1_multiclass[x,3] = 0
        y_test1_multiclass[x,4] = 0
        y_test1_multiclass[x,5] = 0
    
    elif y_test1_temp[x] == 3:
        y_test1_multiclass[x,0] = 0
        y_test1_multiclass[x,1] = 0
        y_test1_multiclass[x,2] = 0
        y_test1_multiclass[x,3] = 1
        y_test1_multiclass[x,4] = 0
        y_test1_multiclass[x,5] = 0

    elif y_test1_temp[x] == 4:
        y_test1_multiclass[x,0] = 0
        y_test1_multiclass[x,1] = 0
        y_test1_multiclass[x,2] = 0
        y_test1_multiclass[x,3] = 0
        y_test1_multiclass[x,4] = 1
        y_test1_multiclass[x,5] = 0

    elif y_test1_temp[x] == 5:
        y_test1_multiclass[x,0] = 0
        y_test1_multiclass[x,1] = 0
        y_test1_multiclass[x,2] = 0
        y_test1_multiclass[x,3] = 0
        y_test1_multiclass[x,4] = 0
        y_test1_multiclass[x,5] = 1

In [ ]:
# Calculate Precision, Recall and Thresholds 
precision = dict()
recall = dict()
average_precision = dict()
for i in range(6):
    precision[i], recall[i], _ = precision_recall_curve(y_test1_multiclass[:, i], y_pred1_multiclass[:, i])
    average_precision[i] = average_precision_score(y_test1_multiclass[:, i], y_pred1_multiclass[:, i])

precision["micro"], recall["micro"], _ = precision_recall_curve(
    y_test1_multiclass.ravel(), y_pred1_multiclass.ravel()
)
average_precision["micro"] = average_precision_score(y_test1_multiclass, y_pred1_multiclass, average="micro")
average_precision['macro'] = (average_precision[0] + average_precision[1] + average_precision[2] + average_precision[3] + average_precision[4] + average_precision[5])/6

In [ ]:
# Visualize the Precision-Recall Curves, add f1 curves for reference 
labely = []
labely.append('V/V & No F508del Mutation')
labely.append('V/M & No F508del Mutation')
labely.append('M/M & No F508del Mutation')
labely.append('M/M & Heterozygous F508del Mutation')
labely.append('M/M & Homozygous F508del Mutation')
labely.append('V/M & Heterozygous F508del Mutation')

colors = cycle(['skyblue', 'orange', 'green', 'red', 'violet', 'blue'])

_, ax = plt.subplots(figsize=(11, 11))

f_scores = np.linspace(0.2, 0.8, num=4)
lines, labels = [], []
for f_score in f_scores:
    x = np.linspace(0.01, 1)
    y = f_score * x / (2 * x - f_score)
    (l,) = plt.plot(x[y >= 0], y[y >= 0], color="gray", alpha=0.2)
    plt.annotate("f1={0:0.1f}".format(f_score), xy=(0.9, y[45] + 0.02))

display = PrecisionRecallDisplay(
    recall=recall["micro"],
    precision=precision["micro"],
    average_precision=average_precision["micro"],
)
display.plot(ax=ax, name="Micro-average of all classes", color="gold")

for i, color in zip(range(6), colors):
    display = PrecisionRecallDisplay(
        recall=recall[i],
        precision=precision[i],
        average_precision= average_precision[i],
    )
    display.plot(ax=ax, name=f"{labely[i]}", color=color)

handles, labels = display.ax_.get_legend_handles_labels()
handles.extend([l])
labels.extend(["iso-f1 curves"])

ax.legend(handles=handles, labels=labels, loc="lower left", fontsize='large')
ax.set_xlabel('Recall', fontsize='x-large', fontweight='bold')
ax.set_ylabel('Precision', fontsize='x-large', fontweight='bold')
custom_ticks = [0, 0.2, 0.4, 0.6, 0.8, 1]
ax.set_xticklabels(custom_ticks,fontsize='x-large')
ax.set_yticklabels(custom_ticks,fontsize='x-large')
plt.xlim(0, 1)
plt.ylim(0, 1.001)
#ax.set_title(f'Intron {Intron}: precision vs. recall curve for {mutation} Mutation Prediction')

filename = f'Intron {Intron} | V470M+F508del | PR Curve.pdf'
file_path = os.path.join(directory_name, filename)
plt.savefig(file_path, format='pdf', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# Tabulate the Average Precision of each class 
PRAUC_table = pd.DataFrame(columns= ['Macro AP', 'Micro AP', 'AP for V/V & No F508del Mutation', 'AP for V/M & No F508del Mutation', 'AP for M/M & No F508del Mutation',
                                     'AP for M/M & Heterozygous F508del Mutation', 'AP for M/M & Homozygous F508del Mutation', 'AP for V/M & Heterozygous F508del Mutation'], index= [f'Intron {Intron}'])

PRAUC_table.loc[f'Intron {Intron}', 'Micro AP'] = average_precision['micro']
PRAUC_table.loc[f'Intron {Intron}', 'AP for V/V & No F508del Mutation'] = average_precision[0]
PRAUC_table.loc[f'Intron {Intron}', 'AP for V/M & No F508del Mutation'] = average_precision[1]
PRAUC_table.loc[f'Intron {Intron}', 'AP for M/M & No F508del Mutation'] = average_precision[2]
PRAUC_table.loc[f'Intron {Intron}', 'AP for M/M & Heterozygous F508del Mutation'] = average_precision[3]
PRAUC_table.loc[f'Intron {Intron}', 'AP for M/M & Homozygous F508del Mutation'] = average_precision[4]
PRAUC_table.loc[f'Intron {Intron}', 'AP for V/M & Heterozygous F508del Mutation'] = average_precision[5]
#PRAUC_table.loc[f'Intron {Intron}', 'Macro Averaged 5 fold CV AP of Training Set'] = round(grid_search.best_score_,3)
PRAUC_table.loc[f'Intron {Intron}', 'Macro AP'] = average_precision['macro']

PRAUC_table = PRAUC_table
PRAUC_table.to_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {Intron} | AP | V470M+F508del.xlsx', index=True)

### Perform a statistical evaluation of Intronic SNP Dosage & Combined V470M and F508del Dosage association 

In [ ]:
# 0.7 Correlation Statistical Evaluation 
important_columns = np.abs(shap_values.values).mean(axis=(0,2)) # calculate mean for each SNP 
important_columns = important_columns > 0 # get all SNPs with a mean SHAP above 0
important_columns_names2 = X_test1.columns[important_columns] # get names for each SNP deemed important by the model (not including associated SNPs)
important_columns_names = Final_SNP_0_7.index # get names for each SNP deemed important by the model (including associated SNPs)

calculated_dosage = pd.DataFrame(index= (important_columns_names), columns=('SNP dosage associated with V/V & No F508del Mutation', 'Odds-Ratio (V/V & No F508del)', 'P-Value (V/V & No F508del)', 'Significant? (V/V & No F508del)',
                                                                           'SNP dosage associated with V/M & No F508del Mutation', 'Odds-Ratio (V/M & No F508del)', 'P-Value (V/M & No F508del)', 'Significant? (V/M & No F508del)',
                                                                           'SNP dosage associated with M/M & No F508del Mutation', 'Odds-Ratio (M/M & No F508del)', 'P-Value (M/M & No F508del)', 'Significant? (M/M & No F508del)',
                                                                           'SNP dosage associated with M/M & Heterozygous F508del Mutation', 'Odds-Ratio (M/M & Heterozygous F508del)', 'P-Value (M/M & Heterozygous F508del)', 'Significant? (M/M & Heterozygous F508del)',
                                                                           'SNP dosage associated with M/M & Homozygous F508del Mutation', 'Odds-Ratio (M/M & Homozygous F508del)', 'P-Value (M/M & Homozygous F508del)', 'Significant? (M/M & Homozygous F508del)',
                                                                           'SNP dosage associated with V/M & Heterozygous F508del Mutation', 'Odds-Ratio (V/M & Heterozygous F508del)', 'P-Value (V/M & Heterozygous F508del)', 'Significant? (V/M & Heterozygous F508del)'))

genotype_order_1 = ['SNP dosage associated with V/V & No F508del Mutation','SNP dosage associated with V/M & No F508del Mutation','SNP dosage associated with M/M & No F508del Mutation',
                    'SNP dosage associated with M/M & Heterozygous F508del Mutation', 'SNP dosage associated with M/M & Homozygous F508del Mutation', 'SNP dosage associated with V/M & Heterozygous F508del Mutation']
genotype_order_2 = ['Odds-Ratio (V/V & No F508del)', 'Odds-Ratio (V/M & No F508del)', 'Odds-Ratio (M/M & No F508del)',
                    'Odds-Ratio (M/M & Heterozygous F508del)','Odds-Ratio (M/M & Homozygous F508del)', 'Odds-Ratio (V/M & Heterozygous F508del)']
genotype_order_3 = ['P-Value (V/V & No F508del)', 'P-Value (V/M & No F508del)', 'P-Value (M/M & No F508del)',
                    'P-Value (M/M & Heterozygous F508del)', 'P-Value (M/M & Homozygous F508del)', 'P-Value (V/M & Heterozygous F508del)']
genotype_order_4 = ['Significant? (V/V & No F508del)', 'Significant? (V/M & No F508del)', 'Significant? (M/M & No F508del)',
                    'Significant? (M/M & Heterozygous F508del)', 'Significant? (M/M & Homozygous F508del)','Significant? (V/M & Heterozygous F508del)']

for feature in important_columns_names:
    feature_data = shap_values

    no_poly= X_test1[feature] == 0 # all individuals with V/V & No F508del
    heterozygous = X_test1[feature] == 0.5 # all individuals with V/M & No F508del Mutation
    homozygous = X_test1[feature] == 1 # all individuals with M/M & No F508del Mutation

    for genotype in range(shap_values.values.shape[2]):

        # If this SNP is associated to a more significant SNP (per the model) assign the dosages as determined by the model for that SNP to this SNP 
        if feature not in important_columns_names2:
            associated_snp = Final_SNP_0_7.loc[feature, 'Associated SNP Locus']
            highest_shap_genotype = calculated_dosage.loc[associated_snp,genotype_order_1[genotype]]

        else:
            # For each genotype, calculate the shap value for each dosage 
            if (shap_values.values[no_poly,X_test1.columns.get_loc(feature), genotype]).size != 0:
                no_poly_shap = np.mean(shap_values.values[no_poly,X_test1.columns.get_loc(feature), genotype])
            else: 
                no_poly_shap = np.nan
            if (shap_values.values[heterozygous,X_test1.columns.get_loc(feature), genotype]).size != 0:
                heterozygous_shap = np.mean(shap_values.values[heterozygous,X_test1.columns.get_loc(feature), genotype])
            else:
                heterozygous_shap = np.nan
            if (shap_values.values[homozygous,X_test1.columns.get_loc(feature), genotype]).size != 0:
                homozygous_shap = np.mean(shap_values.values[homozygous,X_test1.columns.get_loc(feature), genotype])
            else: 
                homozygous_shap = np.nan
            
            # create a key to determine which dosage has the highest mean shap
            shap_dict = {"No Polymorphism": no_poly_shap, "Heterozygous Polymorphism": heterozygous_shap, "Homozygous Polymorphism": homozygous_shap} # order such that equal average mean shaps will yield None as the highest shap genotype 
            highest_shap_genotype = max(shap_dict, key=shap_dict.get) 
            if ((shap_dict[highest_shap_genotype] == 0) or (np.isnan(shap_dict[highest_shap_genotype]))):
                highest_shap_genotype = 'No Prediction'
        
        calculated_dosage.loc[feature, genotype_order_1[genotype]] = highest_shap_genotype
        if highest_shap_genotype == 'No Prediction':
            continue

        # Calculate p-value 
        combined = temp_data1[[feature,'F508del & V470M Combination']] # combine class and genotype
        combined_vals = combined.value_counts() # get count of values 
        
        # Perform Fisher's exact test (greater) tp statistically determine the significance of the association findings 
        if highest_shap_genotype == "No Polymorphism":
            a = combined_vals[(combined_vals.index.get_level_values(0) == 0) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is 0 and where there are V470+F508del Mutation(s) of interest
            b = combined_vals[(combined_vals.index.get_level_values(0) == 0) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is 0 and no V470M+F508del Mutation(s) of interest
            c = combined_vals[(combined_vals.index.get_level_values(0) != 0) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is anything but 0 and there are V470M+F508del Mutation(s) of interest
            d = combined_vals[(combined_vals.index.get_level_values(0) != 0) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is anything but 0 and no V470M+F508del Mutation(s) of interest

        elif highest_shap_genotype == "Heterozygous Polymorphism":
            a = combined_vals[(combined_vals.index.get_level_values(0) == 0.5) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is 0.5 and where there are V470M+F508del Mutation(s) of interest
            b = combined_vals[(combined_vals.index.get_level_values(0) == 0.5) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is 0.5 and no V470M+F508del Mutation(s) of interest
            c = combined_vals[(combined_vals.index.get_level_values(0) != 0.5) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is anything but 0.5 and there are V470M+F508del Mutation(s) of interest
            d = combined_vals[(combined_vals.index.get_level_values(0) != 0.5) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is anything but 0.5 and no V470M+F508del Mutation(s) of interest
         
        elif highest_shap_genotype == "Homozygous Polymorphism":
            a = combined_vals[(combined_vals.index.get_level_values(0) == 1) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is 1 and where there are V470M+F508del Mutation(s) of interest
            b = combined_vals[(combined_vals.index.get_level_values(0) == 1) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is 1 and no V470M+F508del Mutation(s) of interest
            c = combined_vals[(combined_vals.index.get_level_values(0) != 1) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is anything but 1 and there are V470M+F508del Mutation(s) of interest
            d = combined_vals[(combined_vals.index.get_level_values(0) != 1) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is anything but 1 and no V470M+F508del Mutation(s) of interest

        table = np.array([[a,b], [c,d]])
        odds_ratio, p_value = fisher_exact(table,alternative='greater')
        if np.isinf(odds_ratio) or np.isnan(odds_ratio): # inf if b or c == 0 , nan if a or d = 0 while b or c = 0 
            if ((a == 0 and b == 0) or (c == 0 and d == 0) or (a == 0 and c == 0) or (b == 0 and d == 0)):
                odds_ratio = 'Undefined (Unknown Direction)'
            elif (b == 0 or c == 0):
                odds_ratio = '∞ (Strong Positive Association)'
        else:
            odds_ratio = float(f"{odds_ratio:.2f}")

        calculated_dosage.loc[feature, genotype_order_2[genotype]] = odds_ratio
        calculated_dosage.loc[feature, genotype_order_3[genotype]] = f"{p_value:.2e}" if p_value < 0.001 else f"{p_value:.3f}"

        if isinstance(odds_ratio, str):
            if ((odds_ratio == '∞ (Strong Positive Association)') and (float(p_value)< 0.05)):
                calculated_dosage.loc[feature, genotype_order_4[genotype]] = 'yes'
            else: 
                calculated_dosage.loc[feature, genotype_order_4[genotype]] = 'no'
        else: 
            if ((odds_ratio > 1) and (float(p_value) < 0.05)):
                calculated_dosage.loc[feature, genotype_order_4[genotype]] = 'yes'
            else: 
                calculated_dosage.loc[feature, genotype_order_4[genotype]] = 'no'

calculated_dosage.replace(np.nan, '', inplace=True)
Final_SNP_0_7 = Final_SNP_0_7.join(calculated_dosage)
Final_SNP_0_7['main SNP'] = Final_SNP_0_7.apply(lambda x: True if x['Associated SNP Locus'] == '' else False, axis = 1)
Final_SNP_0_7 = Final_SNP_0_7.sort_values(by=['Mean Contribution of Main SNP', 'main SNP', 'Correlation'], ascending = [False, False, False])
Final_SNP_0_7 = Final_SNP_0_7.drop(columns='main SNP')
Final_SNP_0_7.to_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {Intron} | Final SNPs (0.7 Correlation Threshold) | V470M+F508del.xlsx', index=True)
Final_SNP_0_7

## F508del Model

### Set Intron,load data for that Intron and engineer classification for F508del mutations 

In [ ]:
temp_data = pd.read_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/adjusted/Intron_{intron}.xlsx')
temp_data1 = temp_data

### Split Target Feature from Predictor Features

In [ ]:
# Set dependent values for ML F508del
y = temp_data['F508del']

# Set independent Values
X = temp_data1.drop(columns=['F508del', 'V470M'])

### Assess Correlations between Predictor Features

In [ ]:
def correlation_heatmap(train):
    correlations = train.corr(method='spearman')

    fig, ax = plt.subplots(figsize=(20,20))
    mask = np.zeros_like(correlations, dtype=bool)
    mask[np.triu_indices_from(mask)] = True

    cmap1 = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(correlations, vmax=1.0, center = 0, fmt = '.2f', cmap = sns.diverging_palette(230, 20, as_cmap=True), mask = mask, square = True, linewidths = 0.5, annot=True, cbar_kws={"shrink": .70}, annot_kws={'fontweight': 'bold'})

    for i in range(correlations.shape[0]):
        ax.text(i+0.5, i+0.5, 'X', ha='center', va='center', color='red')
    ax.set_xticklabels(ax.get_xticklabels(), fontsize='large', fontweight='bold')
    ax.set_yticklabels(ax.get_yticklabels(), fontsize='large', fontweight='bold')
    plt.tight_layout()
    filename = f'Intron {Intron} | 307 | Correlation.pdf'
    file_path = os.path.join(directory_name, filename)
    plt.savefig(file_path, format='pdf', dpi=600, bbox_inches='tight')
    plt.show()

correlation_heatmap(X)

### Split data into Test and Training sets

In [ ]:
# Split into independent (X) training and testing data, and dependent (Y) training and testing data
X_train1, X_test1, y_train1, y_test1 = train_test_split(X, y,  random_state = random_seed, stratify = y)

# Get counts for each class 
cat_0 = len(y_test1[y_test1==0])
cat_1 = len(y_test1[y_test1==1])
cat_2 = len(y_test1[y_test1==2])

# Confirm if stratification occured (Should be True)
print((abs(len(y_train1[y_train1 == 0])/len(y_train1))-(len(y_test1[y_test1 == 0])/len(y_test1))) <= 0.1)
print((abs(len(y_train1[y_train1 == 1])/len(y_train1))-(len(y_test1[y_test1 == 1])/len(y_test1))) <= 0.1)
print((abs(len(y_train1[y_train1 == 2])/len(y_train1))-(len(y_test1[y_test1 == 2])/len(y_test1))) <= 0.1)

# Check if there are any missing values in the arrays (missing values are fine) 
print(np.any(np.isnan(X_train1)) or np.any(np.isnan(y_train1)) or np.any(np.isnan(X_test1)) or np.any(np.isnan(y_test1)))

# Make sure locus match 
alleles = pd.read_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/adjusted/Intron_{intron}_Alleles.xlsx')
(alleles['locus'] == X.columns).all()

### Access Master file to retrieve Optimal Hyperparameters derived from running GridSearchCV

In [ ]:
mutation_check = 'F508del'

Hyperparameter_master_file = pd.read_excel(f'{workspace_bucket}/data/XGBoost ML Hyperparameters | Master File.xlsx')

index = Hyperparameter_master_file.loc[(Hyperparameter_master_file['Intron'] == Intron) & (Hyperparameter_master_file['Model'] == mutation_check)].index
ae= int(Hyperparameter_master_file.loc[index, 'n_estimators'])
be= int(Hyperparameter_master_file.loc[index, 'max_depth'])
ce= float(Hyperparameter_master_file.loc[index, 'learning_rate'])
de= float(Hyperparameter_master_file.loc[index, 'gamma'])
ee= float(Hyperparameter_master_file.loc[index, 'reg_lambda'])

### Train Model and predict results using the testing dataset

In [ ]:
# fitting for the first dataset (nucleotides as the independent variable), test with the parameters determined in the previous steps 
clf_xgb1 = xgb.XGBClassifier(objective='multi:softprob', eval_metric='aucpr', seed=random_seed, n_estimators = ae
                             , max_depth = be , learning_rate = ce , gamma= de, 
                             reg_lambda = ee) # note, objective set to binary:logistic as this dataset is being used for classification 
# Stopping the tree building early if the evaluation metrics (in this case AUCPR) decreases for 10 rounds in a row 
clf_xgb1.set_params(early_stopping_rounds= 10)
clf_xgb1.fit(X_train1, y_train1, verbose = True, eval_set=[(X_test1, y_test1)])
y_pred1 = clf_xgb1.predict(X_test1)

In [ ]:
# Determine the best round and the aucpr score associated with it 
print('Best Round:', clf_xgb1.best_iteration)
print('AUCPR Score for the Best Round:', clf_xgb1.best_score)

### Interpreting the Model: Mean SHAP Value (% Contribution to the Model)

In [ ]:
# Calculating the Mean Shapley Value for each SNP to each respective associated CFTR Genetic Variant, then calculating what percentage of the CFTR Genetic Variant's prediction is described by the SNP

Mean_abs_shap_df = pd.DataFrame(columns=['Intron', 'F508del Variant','SNP Locus', 'Mean Absolute Shapley Value', '% Contribution of SNP towards F508del Variant Prediction'])
Index = 0
shap_values = shap.Explainer(clf_xgb1).shap_values(X_test1) 

for x in range(3): # range of 3 because of 3 F508del classes
    values = abs(shap_values).mean(0)[:,x] # Obtain shap values for all individuals, depending on class (x)
    sorted_index = np.argsort(values)[::-1] # Retrieve Index based on descending order of the previous values calculated 

    for y in range((values> 0).sum()): # > not really necessary as the absolute value of all shapley values is taken 
        Mean_abs_shap_df.loc[Index, 'Intron'] = Intron
        Mean_abs_shap_df.loc[Index, 'F508del Variant'] = x
        Mean_abs_shap_df.loc[Index, 'SNP Locus'] = X_train1.columns[sorted_index][y]
        Mean_abs_shap_df.loc[Index, 'Mean Absolute Shapley Value'] = values[sorted_index][y]
        Index += 1 # Incremental increases to Index

# Calculate total amount of SHAP values towards predicting each respective class
sum_0 = np.sum(Mean_abs_shap_df[Mean_abs_shap_df['F508del Variant'] == 0]['Mean Absolute Shapley Value']) 
sum_1 = np.sum(Mean_abs_shap_df[Mean_abs_shap_df['F508del Variant'] == 1]['Mean Absolute Shapley Value']) 
sum_2 = np.sum(Mean_abs_shap_df[Mean_abs_shap_df['F508del Variant'] == 2]['Mean Absolute Shapley Value']) 

# Calculate percentage contribution of each SNP towards the class 
for x in range(len(Mean_abs_shap_df)):
    if Mean_abs_shap_df['F508del Variant'][x] == 0:
        Mean_abs_shap_df.loc[x,'% Contribution of SNP towards F508del Variant Prediction'] = ((Mean_abs_shap_df['Mean Absolute Shapley Value'][x])/sum_0)*100 
    elif Mean_abs_shap_df['F508del Variant'][x] == 1:
        Mean_abs_shap_df.loc[x,'% Contribution of SNP towards F508del Variant Prediction'] = ((Mean_abs_shap_df['Mean Absolute Shapley Value'][x])/sum_1)*100
    elif Mean_abs_shap_df['F508del Variant'][x] == 2:
        Mean_abs_shap_df.loc[x,'% Contribution of SNP towards F508del Variant Prediction'] = ((Mean_abs_shap_df['Mean Absolute Shapley Value'][x])/sum_2)*100

Mean_abs_shap_df['F508del Variant'] = Mean_abs_shap_df['F508del Variant'].replace([0,1,2], ["No F508del Mutation", "F508del Heterozygous", "F508del Homozygous"])
Mean_abs_shap_df.to_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {Intron} | Mean Absolute Shapley Values | F508del.xlsx', index=False)
Mean_abs_shap_df

In [ ]:
ax = sns.barplot(Mean_abs_shap_df, errorbar=("pi"), capsize=.1, x='SNP Locus', y='% Contribution of SNP towards F508del Variant Prediction', edgecolor='green', facecolor='white')
x_values = [p.get_text() for p in ax.get_xticklabels()]
y_values = [p.get_height() for p in ax.patches]
allele_values = []

for x in range(len(x_values)):
    for y in range(len(alleles)):
                   if x_values[x] == alleles['locus'][y]:
                    allele_values.append(alleles['alleles'][y])
        
result_df = pd.DataFrame({'SNP Locus': x_values, 'Alleles' : allele_values, 'Mean Contribution': y_values})
result_df.sort_values('Mean Contribution', ascending=False, inplace=True)
result_df.to_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {Intron} | Mean contribution of SNP towards prediction of associated CFTR Genetic Variants | F508del.xlsx', index=False)
result_df

In [ ]:
# Creating a figure which visualizes what visualizes the contribution of each SNP towards prediction in each respective associated class (also shows the mean impact)

fig = plt.figure(figsize=(9,9))
Mean_abs_shap_df.sort_values('% Contribution of SNP towards F508del Variant Prediction', ascending = False, inplace=True)
sns.barplot(Mean_abs_shap_df, errorbar='se', capsize=.1, x='SNP Locus', y='% Contribution of SNP towards F508del Variant Prediction', edgecolor='black', facecolor='white', order=result_df['SNP Locus'], errcolor = 'silver')
plt.yticks(fontsize='xx-large')
plt.xticks(rotation=45, ha='right', fontsize='x-large')
plt.xlabel('SNP Locus', fontsize='x-large', fontweight='bold')
plt.ylabel('% Contribution of SNP towards F508del \n Variant Prediction', fontsize='x-large', fontweight='bold')
sns.swarmplot(Mean_abs_shap_df, x='SNP Locus', y= '% Contribution of SNP towards F508del Variant Prediction', hue='F508del Variant', hue_order=["No F508del Mutation", "F508del Heterozygous", "F508del Homozygous"], edgecolor= 'black', linewidth= 0.5, size = 7.5)
plt.legend(title='F508del Variant', title_fontsize='xx-large', fontsize='xx-large')
plt.tight_layout()
filename = f'Intron {Intron} | F508del | SNP Contribution.pdf'
file_path = os.path.join(directory_name, filename)
plt.savefig(file_path, format='pdf', dpi=600, bbox_inches='tight')
plt.show()

### Determine important SNPs which may be masked due to high correlation

In [ ]:
# Determine Correlation of SNPs deemed important by the model, with other intronic SNPs
correlations = X.corr(method='spearman')
temp_filtered_SNPs = result_df.set_index('SNP Locus')
temp_full_SNPs = alleles.set_index('locus')
correlations_file = pd.DataFrame(columns=('Intron', 'Main SNP Locus', 'Alleles of Main SNP', 'Mean Contribution of main SNP', 'Associated SNP Locus', 'Alleles of Associated SNP', 'Meets Correlation Threshold of 0.9?', 'Correlation'))
index = 0
for x in range(len(x_values)):
    for y in range(len(correlations)):
        if ((correlations[x_values[x]][y] >= 0.7) & (x_values[x] != correlations.index[y])):
            correlations_file.at[index, 'Intron'] = Intron
            correlations_file.at[index, 'Main SNP Locus'] = x_values[x]
            correlations_file.at[index, 'Alleles of Main SNP'] = temp_filtered_SNPs.at[x_values[x], 'Alleles']
            correlations_file.at[index, 'Mean Contribution of main SNP'] = temp_filtered_SNPs.at[x_values[x],'Mean Contribution']
            correlations_file.at[index, 'Associated SNP Locus'] = correlations.index[y]
            correlations_file.at[index, 'Alleles of Associated SNP'] = temp_full_SNPs.at[correlations.index[y], 'alleles']
            correlations_file.at[index, 'Meets Correlation Threshold of 0.9?'] = 'Yes' if correlations[x_values[x]][y] >= 0.9 else 'No'
            correlations_file.at[index, 'Correlation'] = correlations[x_values[x]][y]
            index += 1

correlations_file.to_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {Intron} | SNP Correlations File (filtered for 0.7 threshold) | F508del.xlsx', index='False')   
correlations_file

In [ ]:
# Compile possible SNPS that pass thresholds of 0.7 and 0.9 respectively, for correlation with SNPs important to the model (making sure not to list correlations with other SNPs deemed important by the model)
Final_SNP_Selection = correlations_file[correlations_file['Associated SNP Locus'].isin(result_df['SNP Locus']) == False]

# For 0.7 Threshold
Final_SNP_Selection['Mean Contribution of main SNP'] = pd.to_numeric(
    Final_SNP_Selection['Mean Contribution of main SNP'], errors='coerce')
max_threshold_rows = Final_SNP_Selection.groupby('Associated SNP Locus')['Mean Contribution of main SNP'].idxmax()
Final_SNP_Selection = Final_SNP_Selection.loc[max_threshold_rows]
Final_SNP_Selection = Final_SNP_Selection.reset_index(drop=True)


# For 0.9 Threshold
Final_SNP_Selection2 = Final_SNP_Selection[Final_SNP_Selection['Meets Correlation Threshold of 0.9?'] == 'Yes']
max_threshold_rows2 = Final_SNP_Selection2.groupby('Associated SNP Locus')['Mean Contribution of main SNP'].idxmax()
Final_SNP_Selection2 = Final_SNP_Selection2.loc[max_threshold_rows2]
Final_SNP_Selection2 = Final_SNP_Selection2.reset_index(drop=True)

Final_SNP_Selection2

In [ ]:
# Final SNP table with 0.7 Threshold
Final_SNP_0_7 = result_df
Final_SNP_0_7['Mean Contribution of Main SNP'] = Final_SNP_0_7['Mean Contribution']
Final_SNP_0_7 = Final_SNP_0_7.drop(columns='Mean Contribution')
Final_SNP_0_7['Associated SNP Locus'] = ''
Final_SNP_0_7['Correlation'] = ''
index1 = len(Final_SNP_0_7)

for x in range(len(Final_SNP_Selection)):
    Final_SNP_0_7.at[index1,'SNP Locus'] = Final_SNP_Selection['Associated SNP Locus'][x]
    Final_SNP_0_7.at[index1,'Associated SNP Locus'] = Final_SNP_Selection['Main SNP Locus'][x]
    Final_SNP_0_7.at[index1,'Alleles'] = Final_SNP_Selection['Alleles of Associated SNP'][x]
    Final_SNP_0_7.at[index1,'Mean Contribution of Main SNP'] = Final_SNP_Selection['Mean Contribution of main SNP'][x]
    Final_SNP_0_7.at[index1,'Correlation'] = Final_SNP_Selection['Correlation'][x]
    index1 += 1

Final_SNP_0_7 = Final_SNP_0_7.set_index('SNP Locus')

Final_SNP_0_9 = result_df
Final_SNP_0_9['Mean Contribution of Main SNP'] = Final_SNP_0_9['Mean Contribution']
Final_SNP_0_9 = Final_SNP_0_9.drop(columns='Mean Contribution')
Final_SNP_0_9['Associated SNP Locus'] = ''
Final_SNP_0_9['Correlation'] = ''
index2 = len(Final_SNP_0_9)

# Final SNP table with 0.9 Threshold 
for x in range(len(Final_SNP_Selection2)):
    Final_SNP_0_9.at[index2,'SNP Locus'] = Final_SNP_Selection2['Associated SNP Locus'][x]
    Final_SNP_0_9.at[index2,'Associated SNP Locus'] = Final_SNP_Selection2['Main SNP Locus'][x]
    Final_SNP_0_9.at[index2,'Alleles'] = Final_SNP_Selection2['Alleles of Associated SNP'][x]
    Final_SNP_0_9.at[index2,'Mean Contribution of Main SNP'] = Final_SNP_Selection2['Mean Contribution of main SNP'][x]
    Final_SNP_0_9.at[index2,'Correlation'] = Final_SNP_Selection2['Correlation'][x]
    index2 += 1

Final_SNP_0_9 = Final_SNP_0_9.set_index('SNP Locus')
Final_SNP_0_9

### Interpreting the Model: Mean SHAP Value (contribution to each class)

In [ ]:
shap_values = shap.Explainer(clf_xgb1).shap_values(X_test1)
max_0 = np.sum(abs(shap_values).mean(0)[:,0] > 0 ) + 1
max_1 = np.sum(abs(shap_values).mean(0)[:,1] > 0 ) + 1 
max_2 = np.sum(abs(shap_values).mean(0)[:,2] > 0 ) + 1

labels = ["No F508del Mutation", "F508del Heterozygous", "F508del Homozygous"]

explainer = shap.Explainer(clf_xgb1)
shap_values = explainer(X_test1)

maxy = max(max_0,max_1,max_2) # obtain max number of SNPs amongst each class
max_x_abs = np.max(np.average(np.abs(shap_values.values), axis=(0))) # Obtain SNP importance of the  to set it as x axis limit

plt.subplot(2,3,1)
shap.plots.bar(shap_values[:,:,0], max_display= maxy, show=False)
plt.title(labels[0] + f' (n={cat_0})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='xx-large')
plt.yticks(fontsize='xx-large', fontweight='bold')
plt.xlabel('mean (|SHAP value|)', fontsize='xx-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(0, max_x_abs)

plt.subplot(2,3,2)
shap.plots.bar(shap_values[:,:,1], max_display= maxy, show=False)
plt.title(labels[1] +  f' (n={cat_1})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='xx-large')
plt.yticks(fontsize='xx-large', fontweight='bold')
plt.xlabel('mean (|SHAP value|)', fontsize='xx-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(0, max_x_abs)

plt.subplot(2,3,3)
shap.plots.bar(shap_values[:,:,2], max_display= maxy, show=False)
plt.title(labels[2] + f' (n={cat_2})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='xx-large')
plt.yticks(fontsize='xx-large', fontweight='bold')
plt.xlabel('mean (|SHAP value|)', fontsize='xx-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(0, max_x_abs)

plt.subplots_adjust(top=3, bottom=0.05, left=0.05, right=3.5, hspace=0.3, wspace=0.8)
filename = f'Intron {Intron} | F508del | Barplot.pdf'
file_path = os.path.join(directory_name, filename)
plt.savefig(file_path, format='pdf', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
min_x = np.min(shap_values.values) - (abs(np.min(shap_values.values))*0.01)  # obtain minimum shap value to set x-axis limit, giving slight buffer space aswell
max_x = np.max(shap_values.values) + (abs(np.max(shap_values.values))*0.01) # obtain maximum shap value to set x-axis limit, giving slight buffer space aswell 

plt.subplot(2,3,1)
shap.plots.beeswarm(shap_values[:,:,0], max_display= maxy, color_bar_label='Genotype Call Value of SNP',show=False)
cbar = plt.gcf().axes[-1]
vmin, vmax = cbar.get_ylim()
new_ticks = [vmin, (vmin+vmax)/2, vmax]
new_labels = ['0/0', '0/1\nor\n1/0', '1/1']
cbar.set_yticks(new_ticks)
cbar.set_yticklabels(new_labels)
cbar.yaxis.set_label_coords(9.5, 0.5)
plt.title(labels[0] + f' (n={cat_0})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='xx-large')
plt.yticks(fontsize='xx-large', fontweight='bold')
plt.xlabel('SHAP value', fontsize='xx-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(min_x, max_x)

plt.subplot(2,3,2)
shap.plots.beeswarm(shap_values[:,:,1], max_display= maxy, color_bar_label='Genotype Call Value of SNP',show=False)
cbar = plt.gcf().axes[-1]
vmin, vmax = cbar.get_ylim()
new_ticks = [vmin, (vmin+vmax)/2, vmax]
new_labels = ['0/0', '0/1\nor\n1/0', '1/1']
cbar.set_yticks(new_ticks)
cbar.set_yticklabels(new_labels)
cbar.yaxis.set_label_coords(9.5, 0.5)
plt.title(labels[1] + f' (n={cat_1})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='xx-large')
plt.yticks(fontsize='xx-large', fontweight='bold')
plt.xlabel('SHAP value', fontsize='xx-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(min_x, max_x)

plt.subplot(2,3,3)
shap.plots.beeswarm(shap_values[:,:,2], max_display= maxy, color_bar_label='Genotype Call Value of SNP',show=False)
cbar = plt.gcf().axes[-1]
vmin, vmax = cbar.get_ylim()
new_ticks = [vmin, (vmin+vmax)/2, vmax]
new_labels = ['0/0', '0/1\nor\n1/0', '1/1']
cbar.set_yticks(new_ticks)
cbar.set_yticklabels(new_labels)
cbar.yaxis.set_label_coords(9.5, 0.5)
plt.title(labels[2] + f' (n={cat_2})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='xx-large')
plt.yticks(fontsize='xx-large', fontweight='bold')
plt.xlabel('SHAP value', fontsize='xx-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(min_x, max_x)

plt.subplots_adjust(top=3, bottom=0.05, left=0.05, right=3.5, hspace=0.3, wspace=0.8)
filename = f'Intron {Intron} | F508del | Dotplot.pdf'
file_path = os.path.join(directory_name, filename)
plt.savefig(file_path, format='pdf', dpi=600, bbox_inches='tight')
plt.show()

### Assessing Model Performance: Confusion Matrix

In [ ]:
# Building a confusion matrix 
cm1 = confusion_matrix(y_test1, y_pred1)
cm1_percent = cm1 / (cm1.sum(axis=1)[:, np.newaxis])

disp1 = ConfusionMatrixDisplay(confusion_matrix = cm1, display_labels= ["No F508del", "F508del\nHeterozygous", "F508del\nHomozygous"])
disp2 = ConfusionMatrixDisplay(confusion_matrix = cm1_percent, display_labels= ["No F508del", "F508del\nHeterozygous", "F508del\nHomozygous"])
fig, ax = plt.subplots(1,2, figsize=(24,12))
disp1.plot(ax=ax[0])
disp2.plot(ax=ax[1], values_format='.0%')
disp1.ax_.set_xticklabels(disp1.ax_.get_xticklabels(), rotation=45, ha='right', fontsize=24)
disp1.ax_.set_yticklabels(disp1.ax_.get_yticklabels(), fontsize=24)
disp1.ax_.set_ylabel('True Label', fontsize=24, fontweight='bold')
disp1.ax_.set_xlabel('Predicted Label', fontsize=24, fontweight='bold')
disp2.ax_.set_xticklabels(disp2.ax_.get_xticklabels(), rotation=45, ha='right', fontsize=24)
disp2.ax_.set_yticklabels(disp2.ax_.get_yticklabels(), fontsize=24)
disp2.ax_.set_ylabel('True Label', fontsize=24, fontweight='bold')
disp2.ax_.set_xlabel('Predicted Label', fontsize=24, fontweight='bold')
disp1.ax_.xaxis.labelpad = 16
disp2.ax_.xaxis.labelpad = 16
disp1.ax_.yaxis.labelpad = 16
disp2.ax_.yaxis.labelpad = 16

# Change font size within the confusion matrix
for axis in ax:
    for text in axis.texts:
        text.set_fontsize(26)
        
# Add red borders diagonally 
for i in range(len(cm1)):
    disp1.ax_.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1, fill=False, edgecolor='red', lw=3, zorder = 10))
for i in range(len(cm1_percent)):
    disp2.ax_.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1, fill=False, edgecolor='red', lw=3, zorder = 10))

# Color bar formatting
cbar1 = disp1.im_.colorbar
cbar1.ax.yaxis.label.set_size(22)
cbar1.ax.yaxis.set_tick_params(labelsize=22)

cbar2 = disp2.im_.colorbar
cbar2.ax.yaxis.label.set_size(22)
cbar2.ax.yaxis.set_tick_params(labelsize=22)

    
plt.subplots_adjust(wspace=0.42)
filename = f'Intron {Intron} | F508del | Confusion Matrix.pdf'
file_path = os.path.join(directory_name, filename)
#fig.suptitle(f'Confusion Matrix of {mutation} Mutation Classification for Intron {Intron} ', y = 0.92)
plt.savefig(file_path, format='pdf', dpi=600, bbox_inches='tight')
fig.show()

### Assessing Model Performance: Precision-Recall Curves

In [ ]:
# Convert y_test1 to the same format as y_pred1 (rather than having an array of shape [77,1] have an array with shape [77,3] with each column depicting which class the individual is)
y_pred1 = clf_xgb1.predict_proba(X_test1)
y_pred1_multiclass = clf_xgb1.predict_proba(X_test1)
y_test1_temp = y_test1.reset_index(drop=True).copy()
y_test1_multiclass = np.zeros((y_pred1.shape[0],y_pred1.shape[1]))

for x in range(y_test1_temp.shape[0]):
    if y_test1_temp[x] == 0:
        y_test1_multiclass[x,0] = 1
        y_test1_multiclass[x,1] = 0
        y_test1_multiclass[x,2] = 0


    elif y_test1_temp[x] == 1:
        y_test1_multiclass[x,0] = 0
        y_test1_multiclass[x,1] = 1
        y_test1_multiclass[x,2] = 0


    elif y_test1_temp[x] == 2:
        y_test1_multiclass[x,0] = 0
        y_test1_multiclass[x,1] = 0
        y_test1_multiclass[x,2] = 1

In [ ]:
# Calculate Precision, Recall and Thresholds 
precision = dict()
recall = dict()
average_precision = dict()
for i in range(3):
    precision[i], recall[i], _ = precision_recall_curve(y_test1_multiclass[:, i], y_pred1_multiclass[:, i])
    average_precision[i] = average_precision_score(y_test1_multiclass[:, i], y_pred1_multiclass[:, i])

precision["micro"], recall["micro"], _ = precision_recall_curve(
    y_test1_multiclass.ravel(), y_pred1_multiclass.ravel()
)
average_precision["micro"] = average_precision_score(y_test1_multiclass, y_pred1_multiclass, average="micro")
average_precision['macro'] = (average_precision[0] + average_precision[1] + average_precision[2])/3

In [ ]:
# Visualize the Precision-Recall Curves, add f1 curves for reference 
labely = []
labely.append('No F508del Mutation')
labely.append('F508del Heterozygous')
labely.append('F508del Homozygous')
colors = cycle(['blue', 'orange', 'green'])

_, ax = plt.subplots(figsize=(11, 11))

f_scores = np.linspace(0.2, 0.8, num=4)
lines, labels = [], []
for f_score in f_scores:
    x = np.linspace(0.01, 1)
    y = f_score * x / (2 * x - f_score)
    (l,) = plt.plot(x[y >= 0], y[y >= 0], color="gray", alpha=0.2)
    plt.annotate("f1={0:0.1f}".format(f_score), xy=(0.9, y[45] + 0.02), fontsize = 15)

display = PrecisionRecallDisplay(
    recall=recall["micro"],
    precision=precision["micro"],
    average_precision=average_precision["micro"],
)
display.plot(ax=ax, name="Micro-average of all classes", color="gold")

for i, color in zip(range(3), colors):
    display = PrecisionRecallDisplay(
        recall=recall[i],
        precision=precision[i],
        average_precision= average_precision[i],
    )
    display.plot(ax=ax, name=f"{labely[i]}", color=color)

handles, labels = display.ax_.get_legend_handles_labels()
handles.extend([l])
labels.extend(["iso-f1 curves"])

ax.legend(handles=handles, labels=labels, loc="lower left", fontsize='xx-large')
ax.set_xlabel('Recall', fontsize='xx-large', fontweight='bold')
ax.set_ylabel('Precision', fontsize='xx-large', fontweight='bold')
custom_ticks = [0, 0.2, 0.4, 0.6, 0.8, 1]
ax.set_xticklabels(custom_ticks,fontsize='xx-large')
ax.set_yticklabels(custom_ticks,fontsize='xx-large')
plt.xlim(0, 1.005)
plt.ylim(0, 1.005)
#ax.set_title(f'Intron {Intron}: precision vs. recall curve for {mutation} Mutation Prediction')

filename = f'Intron {Intron} | F508del | PR Curve.pdf'
file_path = os.path.join(directory_name, filename)
plt.savefig(file_path, format='pdf', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# Tabulate the Average Precision of each class 
PRAUC_table = pd.DataFrame(columns= ['Macro Averaged AP', 'Micro Averaged AP', 'AP for No F508del Mutation', 'AP for F508del Heterozygous', 'AP for F508del Homozygous'], index= [f'Intron {Intron}'])

PRAUC_table.loc[f'Intron {Intron}', 'Micro Averaged AP'] = average_precision['micro']
PRAUC_table.loc[f'Intron {Intron}', 'AP for No F508del Mutation'] = average_precision[0]
PRAUC_table.loc[f'Intron {Intron}', 'AP for F508del Heterozygous'] = average_precision[1]
PRAUC_table.loc[f'Intron {Intron}', 'AP for F508del Homozygous'] = average_precision[2]
#PRAUC_table.loc[f'Intron {Intron}', 'Macro Averaged 5 fold CV AP of Training Set'] = round(grid_search.best_score_,3)
PRAUC_table.loc[f'Intron {Intron}', 'Macro Averaged AP'] = average_precision['macro']

PRAUC_table.to_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {Intron} | AP | F508del.xlsx', index=True)

### Perform a statistical evaluation of Intronic SNP Dosage & F508del Mutation Dosage association 

In [ ]:
# 0.7 Correlation Statistical Evaluation 
important_columns = np.abs(shap_values.values).mean(axis=(0,2)) # calculate mean for each SNP 
important_columns = important_columns > 0 # get all SNPs with a mean SHAP above 0
important_columns_names2 = X_test1.columns[important_columns] # get names for each SNP deemed important by the model (not including associated SNPs)
important_columns_names = Final_SNP_0_7.index # get names for each SNP deemed important by the model (including associated SNPs)

calculated_dosage = pd.DataFrame(index= (important_columns_names), columns=('SNP dosage associated with No F508del Mutation', 'Odds-Ratio (No F508del)', 'P-Value (No F508del)', 'Significant? (No F508del)',
                                                                           'SNP dosage associated with Heterozygous F508del Mutation', 'Odds-Ratio (F508del Heterozygous)', 'P-Value (F508del Heterozygous)', 'Significant? (F508del Heterozygous)',
                                                                           'SNP dosage associated with Homozygous F508del Mutation', 'Odds-Ratio (F508del Homozygous)', 'P-Value (F508del Homozygous)', 'Significant? (F508del Homozygous)'))

genotype_order_1 = ['SNP dosage associated with No F508del Mutation','SNP dosage associated with Heterozygous F508del Mutation','SNP dosage associated with Homozygous F508del Mutation']
genotype_order_2 = ['Odds-Ratio (No F508del)', 'Odds-Ratio (F508del Heterozygous)', 'Odds-Ratio (F508del Homozygous)']
genotype_order_3 = ['P-Value (No F508del)', 'P-Value (F508del Heterozygous)', 'P-Value (F508del Homozygous)']
genotype_order_4 = ['Significant? (No F508del)', 'Significant? (F508del Heterozygous)', 'Significant? (F508del Homozygous)']

for feature in important_columns_names:
    no_poly= X_test1[feature] == 0 # all individuals with no mutation
    heterozygous = X_test1[feature] == 0.5 # all individuals who are heterozygous
    homozygous = X_test1[feature] == 1 # all individuals who are homozygous

    for genotype in range(shap_values.values.shape[2]):

        # If this SNP is associated to a more significant SNP (per the model) assign the dosages as determined by the model for that SNP to this SNP 
        if feature not in important_columns_names2:
            associated_snp = Final_SNP_0_7.loc[feature, 'Associated SNP Locus']
            highest_shap_genotype = calculated_dosage.loc[associated_snp,genotype_order_1[genotype]]

        else:
            # For each genotype, calculate the shap value for each dosage 
            if (shap_values.values[no_poly,X_test1.columns.get_loc(feature), genotype]).size != 0:
                no_poly_shap = np.mean(shap_values.values[no_poly,X_test1.columns.get_loc(feature), genotype])
            else: 
                no_poly_shap = np.nan
            if (shap_values.values[heterozygous,X_test1.columns.get_loc(feature), genotype]).size != 0:
                heterozygous_shap = np.mean(shap_values.values[heterozygous,X_test1.columns.get_loc(feature), genotype])
            else:
                heterozygous_shap = np.nan
            if (shap_values.values[homozygous,X_test1.columns.get_loc(feature), genotype]).size != 0:
                homozygous_shap = np.mean(shap_values.values[homozygous,X_test1.columns.get_loc(feature), genotype])
            else: 
                homozygous_shap = np.nan
            
            # create a key to determine which dosage has the highest mean shap
            shap_dict = {"No Polymorphism": no_poly_shap, "Heterozygous Polymorphism": heterozygous_shap, "Homozygous Polymorphism": homozygous_shap} # order such that equal average mean shaps will yield None as the highest shap genotype 
            highest_shap_genotype = max(shap_dict, key=shap_dict.get) 
            if ((shap_dict[highest_shap_genotype] == 0) or (np.isnan(shap_dict[highest_shap_genotype]))):
                highest_shap_genotype = 'No Prediction'
        
        calculated_dosage.loc[feature, genotype_order_1[genotype]] = highest_shap_genotype
        if highest_shap_genotype == 'No Prediction':
            continue

        # Calculate p-value 
        combined = temp_data[[feature,'F508del']]
        combined_vals = combined.value_counts() # get count of values 
        
        # Perform Fisher's exact test (greater) tp statistically determine the significance of the association findings 
        if highest_shap_genotype == "No Polymorphism":
            a = combined_vals[(combined_vals.index.get_level_values(0) == 0) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is 0 and where there is F508del genotype of interest
            b = combined_vals[(combined_vals.index.get_level_values(0) == 0) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is 0 and no F508del genotype of interest
            c = combined_vals[(combined_vals.index.get_level_values(0) != 0) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is anything but 0 and there is F508del genotype of interest
            d = combined_vals[(combined_vals.index.get_level_values(0) != 0) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is anything but 0 and no F508del genotype of interest

        elif highest_shap_genotype == "Heterozygous Polymorphism":
            a = combined_vals[(combined_vals.index.get_level_values(0) == 0.5) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is 0.5 and where there is F508del genotype of interest
            b = combined_vals[(combined_vals.index.get_level_values(0) == 0.5) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is 0.5 and no F508del genotype of interest
            c = combined_vals[(combined_vals.index.get_level_values(0) != 0.5) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is anything but 0.5 and there is F508del genotype of interest
            d = combined_vals[(combined_vals.index.get_level_values(0) != 0.5) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is anything but 0.5 and no F508del genotype of interest
         
        elif highest_shap_genotype == "Homozygous Polymorphism":
            a = combined_vals[(combined_vals.index.get_level_values(0) == 1) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is 1 and where there is F508del genotype of interest
            b = combined_vals[(combined_vals.index.get_level_values(0) == 1) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is 1 and no F508del genotype of interest
            c = combined_vals[(combined_vals.index.get_level_values(0) != 1) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is anything but 1 and there is F508del genotype of interest
            d = combined_vals[(combined_vals.index.get_level_values(0) != 1) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is anything but 1 and no F508del genotype of interest

    
        table = np.array([[a,b], [c,d]])
        odds_ratio, p_value = fisher_exact(table,alternative='greater')
        if np.isinf(odds_ratio) or np.isnan(odds_ratio): # inf if b or c == 0 , nan if a or d = 0 while b or c = 0 
            if ((a == 0 and b == 0) or (c == 0 and d == 0) or (a == 0 and c == 0) or (b == 0 and d == 0)):
                odds_ratio = 'Undefined (Unknown Direction)'
            elif (b == 0 or c == 0):
                odds_ratio = '∞ (Strong Positive Association)'
        else:
            odds_ratio = float(f"{odds_ratio:.2f}")

        calculated_dosage.loc[feature, genotype_order_2[genotype]] = odds_ratio
        calculated_dosage.loc[feature, genotype_order_3[genotype]] = f"{p_value:.2e}" if p_value < 0.001 else f"{p_value:.3f}"

        if isinstance(odds_ratio, str):
            if ((odds_ratio == '∞ (Strong Positive Association)') and (float(p_value)< 0.05)):
                calculated_dosage.loc[feature, genotype_order_4[genotype]] = 'yes'
            else: 
                calculated_dosage.loc[feature, genotype_order_4[genotype]] = 'no'
        else: 
            if ((odds_ratio > 1) and (float(p_value) < 0.05)):
                calculated_dosage.loc[feature, genotype_order_4[genotype]] = 'yes'
            else: 
                calculated_dosage.loc[feature, genotype_order_4[genotype]] = 'no'

calculated_dosage.replace(np.nan, '', inplace=True)
Final_SNP_0_7 = Final_SNP_0_7.join(calculated_dosage)
Final_SNP_0_7['main SNP'] = Final_SNP_0_7.apply(lambda x: True if x['Associated SNP Locus'] == '' else False, axis = 1)
Final_SNP_0_7 = Final_SNP_0_7.sort_values(by=['Mean Contribution of Main SNP', 'main SNP', 'Correlation'], ascending = [False, False, False])
Final_SNP_0_7 = Final_SNP_0_7.drop(columns='main SNP')
Final_SNP_0_7.to_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {Intron} | Final SNPs (0.7 Correlation Threshold) | F508del.xlsx', index=True)
Final_SNP_0_7


## V470M Model

### Set Intron,load data for that Intron and engineer classification for V470M mutations

In [ ]:
temp_data = pd.read_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/adjusted/Intron_{intron}.xlsx')
temp_data = temp_data[pd.notna(temp_data['V470M'])]
temp_data1 = temp_data

In [ ]:
# Set dependent values for ML V470M
y = temp_data['V470M']

# Set independent Values
X = temp_data1.drop(columns=['F508del', 'V470M'])

### Assess Correlations between Predictor Features

In [ ]:
def correlation_heatmap(train):
    correlations = train.corr(method='spearman')

    fig, ax = plt.subplots(figsize=(20,20))
    mask = np.zeros_like(correlations, dtype=bool)
    mask[np.triu_indices_from(mask)] = True

    cmap1 = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(correlations, vmax=1.0, center = 0, fmt = '.2f', cmap = sns.diverging_palette(230, 20, as_cmap=True), mask = mask, square = True, linewidths = 0.5, annot=True, cbar_kws={"shrink": .70}, annot_kws={'fontweight': 'bold'})

    for i in range(correlations.shape[0]):
        ax.text(i+0.5, i+0.5, 'X', ha='center', va='center', color='red')
    ax.set_xticklabels(ax.get_xticklabels(), fontsize='large', fontweight='bold')
    ax.set_yticklabels(ax.get_yticklabels(), fontsize='large', fontweight='bold')
    plt.tight_layout()
    filename = f'Intron {Intron} | 306 | Correlation.pdf'
    file_path = os.path.join(directory_name, filename)
    plt.savefig(file_path, format='pdf', dpi=600, bbox_inches='tight')
    plt.show()

correlation_heatmap(X)

### Split data into Test and Training sets

In [ ]:
# Split into independent (X) training and testing data, and dependent (Y) training and testing data
X_train1, X_test1, y_train1, y_test1 = train_test_split(X, y,  random_state = random_seed, stratify = y)

# Get counts for each class 
cat_0 = len(y_test1[y_test1==0])
cat_1 = len(y_test1[y_test1==1])
cat_2 = len(y_test1[y_test1==2])

# Confirm if stratification occured (Should be True)
print((abs(len(y_train1[y_train1 == 0])/len(y_train1))-(len(y_test1[y_test1 == 0])/len(y_test1))) <= 0.1)
print((abs(len(y_train1[y_train1 == 1])/len(y_train1))-(len(y_test1[y_test1 == 1])/len(y_test1))) <= 0.1)
print((abs(len(y_train1[y_train1 == 2])/len(y_train1))-(len(y_test1[y_test1 == 2])/len(y_test1))) <= 0.1)

# Check if there are any missing values in the arrays (missing values are fine) 
print(np.any(np.isnan(X_train1)) or np.any(np.isnan(y_train1)) or np.any(np.isnan(X_test1)) or np.any(np.isnan(y_test1)))

# Make sure locus match 
alleles = pd.read_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/adjusted/Intron_{intron}_Alleles.xlsx')
(alleles['locus'] == X.columns).all()

### Access Master file to retrieve Optimal Hyperparameters derived from running GridSearchCV

In [ ]:
mutation_check = 'V470M'

Hyperparameter_master_file = pd.read_excel(f'{workspace_bucket}/data/XGBoost ML Hyperparameters | Master File.xlsx')

index = Hyperparameter_master_file.loc[(Hyperparameter_master_file['Intron'] == Intron) & (Hyperparameter_master_file['Model'] == mutation_check)].index
ae= int(Hyperparameter_master_file.loc[index, 'n_estimators'])
be= int(Hyperparameter_master_file.loc[index, 'max_depth'])
ce= float(Hyperparameter_master_file.loc[index, 'learning_rate'])
de= float(Hyperparameter_master_file.loc[index, 'gamma'])
ee= float(Hyperparameter_master_file.loc[index, 'reg_lambda'])

### Train Model and predict results using the testing dataset

In [ ]:
# fitting for the first dataset (nucleotides as the independent variable), test with the parameters determined in the previous steps 
clf_xgb1 = xgb.XGBClassifier(objective='multi:softprob', eval_metric='aucpr', seed=random_seed, n_estimators = ae
                             , max_depth = be , learning_rate = ce , gamma= de, 
                             reg_lambda = ee) # note, objective set to binary:logistic as this dataset is being used for classification 
# Stopping the tree building early if the evaluation metrics (in this case AUCPR) decreases for 10 rounds in a row 
clf_xgb1.set_params(early_stopping_rounds= 10)
clf_xgb1.fit(X_train1, y_train1, verbose = True, eval_set=[(X_test1, y_test1)])
y_pred1 = clf_xgb1.predict(X_test1)

In [ ]:
# Determine the best round and the aucpr score associated with it 
print('Best Round:', clf_xgb1.best_iteration)
print('AUCPR Score for the Best Round:', clf_xgb1.best_score)

### Interpreting the Model: Mean SHAP Value (% Contribution to the Model)

In [ ]:
# Calculating the Mean Shapley Value for each SNP to each respective associated CFTR Genetic Variant, then calculating what percentage of the CFTR Genetic Variant's prediction is described by the SNP

Mean_abs_shap_df = pd.DataFrame(columns=['Intron', 'V470M Variant','SNP Locus', 'Mean Absolute Shapley Value', '% Contribution of SNP towards V470M Variant Prediction'])
Index = 0
shap_values = shap.Explainer(clf_xgb1).shap_values(X_test1)

for x in range(3): # range of 3 because of 3 V470M classes
    values = abs(shap_values).mean(0)[:,x] # Obtain shap values for all individuals, depending on class (x)
    sorted_index = np.argsort(values)[::-1] # Retrieve Index based on descending order of the previous values calculated 

    for y in range((values> 0).sum()): # > not really necessary as the absolute value of all shapley values is taken 
        Mean_abs_shap_df.loc[Index, 'Intron'] = Intron
        Mean_abs_shap_df.loc[Index, 'V470M Variant'] = x
        Mean_abs_shap_df.loc[Index, 'SNP Locus'] = X_train1.columns[sorted_index][y]
        Mean_abs_shap_df.loc[Index, 'Mean Absolute Shapley Value'] = values[sorted_index][y]
        Index += 1 # Incremental increases to Index

# Calculate total amount of SHAP values towards predicting each respective class
sum_0 = np.sum(Mean_abs_shap_df[Mean_abs_shap_df['V470M Variant'] == 0]['Mean Absolute Shapley Value'])
sum_1 = np.sum(Mean_abs_shap_df[Mean_abs_shap_df['V470M Variant'] == 1]['Mean Absolute Shapley Value'])
sum_2 = np.sum(Mean_abs_shap_df[Mean_abs_shap_df['V470M Variant'] == 2]['Mean Absolute Shapley Value'])

# Calculate percentage contribution of each SNP towards the class 
for x in range(len(Mean_abs_shap_df)):
    if Mean_abs_shap_df['V470M Variant'][x] == 0:
        Mean_abs_shap_df.loc[x,'% Contribution of SNP towards V470M Variant Prediction'] = ((Mean_abs_shap_df['Mean Absolute Shapley Value'][x])/sum_0)*100
    elif Mean_abs_shap_df['V470M Variant'][x] == 1:
        Mean_abs_shap_df.loc[x,'% Contribution of SNP towards V470M Variant Prediction'] = ((Mean_abs_shap_df['Mean Absolute Shapley Value'][x])/sum_1)*100
    elif Mean_abs_shap_df['V470M Variant'][x] == 2:
        Mean_abs_shap_df.loc[x,'% Contribution of SNP towards V470M Variant Prediction'] = ((Mean_abs_shap_df['Mean Absolute Shapley Value'][x])/sum_2)*100

Mean_abs_shap_df['V470M Variant'] = Mean_abs_shap_df['V470M Variant'].replace([0,1,2], ["V/V", "V/M", "M/M"])
Mean_abs_shap_df.to_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {Intron} | Mean Absolute Shapley Values | V470M.xlsx', index=False)
Mean_abs_shap_df

In [ ]:
ax = sns.barplot(Mean_abs_shap_df, errorbar=("pi"), capsize=.1, x='SNP Locus', y='% Contribution of SNP towards V470M Variant Prediction', edgecolor='green', facecolor='white')
x_values = [p.get_text() for p in ax.get_xticklabels()]
y_values = [p.get_height() for p in ax.patches]
allele_values = []

for x in range(len(x_values)):
    for y in range(len(alleles)):
                   if x_values[x] == alleles['locus'][y]:
                    allele_values.append(alleles['alleles'][y])
        
result_df = pd.DataFrame({'SNP Locus': x_values, 'Alleles' : allele_values, 'Mean Contribution': y_values})
result_df.sort_values('Mean Contribution', ascending=False, inplace=True)
result_df.to_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {Intron} | Mean contribution of SNP towards prediction of associated CFTR Genetic Variants | V470M.xlsx', index=False)
result_df

In [ ]:
# Creating a figure which visualizes what visualizes the contribution of each SNP towards prediction in each respective associated class (also shows the mean impact)

fig = plt.figure(figsize=(9,9))
Mean_abs_shap_df.sort_values('% Contribution of SNP towards V470M Variant Prediction', ascending = False, inplace=True)
sns.barplot(Mean_abs_shap_df, errorbar='se', capsize=.1, x='SNP Locus', y='% Contribution of SNP towards V470M Variant Prediction', edgecolor='black', facecolor='white', order= result_df['SNP Locus'], errcolor = 'silver')
plt.yticks(fontsize='xx-large')
plt.xticks(rotation=45, ha='right', fontsize='x-large')
plt.xlabel('SNP Locus', fontsize='x-large', fontweight='bold')
plt.ylabel('% Contribution of SNP towards V470M \n Variant Prediction', fontsize='x-large', fontweight='bold')
sns.swarmplot(Mean_abs_shap_df, x='SNP Locus', y= '% Contribution of SNP towards V470M Variant Prediction', hue='V470M Variant', hue_order=["V/V", "V/M", "M/M"], edgecolor= 'black', linewidth= 0.5, size= 7.5)
plt.legend(title='V470M Variant', title_fontsize='xx-large', fontsize='xx-large')
plt.tight_layout()
filename = f'Intron {Intron} | V470M | SNP Contribution.pdf'
file_path = os.path.join(directory_name, filename)
plt.savefig(file_path, format='pdf', dpi=600, bbox_inches='tight')
plt.show()

### Determine important SNPs which may be masked due to high correlation

In [ ]:
# Determine Correlation of SNPs deemed important by the model, with other intronic SNPs
correlations = X.corr(method='spearman')
temp_filtered_SNPs = result_df.set_index('SNP Locus')
temp_full_SNPs = alleles.set_index('locus')
correlations_file = pd.DataFrame(columns=('Intron', 'Main SNP Locus', 'Alleles of Main SNP', 'Mean Contribution of main SNP', 'Associated SNP Locus', 'Alleles of Associated SNP', 'Meets Correlation Threshold of 0.9?', 'Correlation'))
index = 0
for x in range(len(x_values)):
    for y in range(len(correlations)):
        if ((correlations[x_values[x]][y] >= 0.7) & (x_values[x] != correlations.index[y])):
            correlations_file.at[index, 'Intron'] = Intron
            correlations_file.at[index, 'Main SNP Locus'] = x_values[x]
            correlations_file.at[index, 'Alleles of Main SNP'] = temp_filtered_SNPs.at[x_values[x], 'Alleles']
            correlations_file.at[index, 'Mean Contribution of main SNP'] = temp_filtered_SNPs.at[x_values[x],'Mean Contribution']
            correlations_file.at[index, 'Associated SNP Locus'] = correlations.index[y]
            correlations_file.at[index, 'Alleles of Associated SNP'] = temp_full_SNPs.at[correlations.index[y], 'alleles']
            correlations_file.at[index, 'Meets Correlation Threshold of 0.9?'] = 'Yes' if correlations[x_values[x]][y] >= 0.9 else 'No'
            correlations_file.at[index, 'Correlation'] = correlations[x_values[x]][y]
            index += 1

correlations_file.to_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {Intron} | SNP Correlations File (filtered for 0.7 threshold) | V470M.xlsx', index='False')   
correlations_file

In [ ]:
# Compile possible SNPS that pass thresholds of 0.7 and 0.9 respectively, for correlation with SNPs important to the model (making sure not to list correlations with other SNPs deemed important by the model)
Final_SNP_Selection = correlations_file[correlations_file['Associated SNP Locus'].isin(result_df['SNP Locus']) == False]

# For 0.7 Threshold
Final_SNP_Selection['Mean Contribution of main SNP'] = pd.to_numeric(
    Final_SNP_Selection['Mean Contribution of main SNP'], errors='coerce')
max_threshold_rows = Final_SNP_Selection.groupby('Associated SNP Locus')['Mean Contribution of main SNP'].idxmax()
Final_SNP_Selection = Final_SNP_Selection.loc[max_threshold_rows]
Final_SNP_Selection = Final_SNP_Selection.reset_index(drop=True)


# For 0.9 Threshold
Final_SNP_Selection2 = Final_SNP_Selection[Final_SNP_Selection['Meets Correlation Threshold of 0.9?'] == 'Yes']
max_threshold_rows2 = Final_SNP_Selection2.groupby('Associated SNP Locus')['Mean Contribution of main SNP'].idxmax()
Final_SNP_Selection2 = Final_SNP_Selection2.loc[max_threshold_rows2]
Final_SNP_Selection2 = Final_SNP_Selection2.reset_index(drop=True)

Final_SNP_Selection2

In [ ]:
# Final SNP table with 0.7 Threshold
Final_SNP_0_7 = result_df
Final_SNP_0_7['Mean Contribution of Main SNP'] = Final_SNP_0_7['Mean Contribution']
Final_SNP_0_7 = Final_SNP_0_7.drop(columns='Mean Contribution')
Final_SNP_0_7['Associated SNP Locus'] = ''
Final_SNP_0_7['Correlation'] = ''
index1 = len(Final_SNP_0_7)

for x in range(len(Final_SNP_Selection)):
    Final_SNP_0_7.at[index1,'SNP Locus'] = Final_SNP_Selection['Associated SNP Locus'][x]
    Final_SNP_0_7.at[index1,'Associated SNP Locus'] = Final_SNP_Selection['Main SNP Locus'][x]
    Final_SNP_0_7.at[index1,'Alleles'] = Final_SNP_Selection['Alleles of Associated SNP'][x]
    Final_SNP_0_7.at[index1,'Mean Contribution of Main SNP'] = Final_SNP_Selection['Mean Contribution of main SNP'][x]
    Final_SNP_0_7.at[index1,'Correlation'] = Final_SNP_Selection['Correlation'][x]
    index1 += 1

Final_SNP_0_7 = Final_SNP_0_7.set_index('SNP Locus')

Final_SNP_0_9 = result_df
Final_SNP_0_9['Mean Contribution of Main SNP'] = Final_SNP_0_9['Mean Contribution']
Final_SNP_0_9 = Final_SNP_0_9.drop(columns='Mean Contribution')
Final_SNP_0_9['Associated SNP Locus'] = ''
Final_SNP_0_9['Correlation'] = ''
index2 = len(Final_SNP_0_9)

# Final SNP table with 0.9 Threshold 
for x in range(len(Final_SNP_Selection2)):
    Final_SNP_0_9.at[index2,'SNP Locus'] = Final_SNP_Selection2['Associated SNP Locus'][x]
    Final_SNP_0_9.at[index2,'Associated SNP Locus'] = Final_SNP_Selection2['Main SNP Locus'][x]
    Final_SNP_0_9.at[index2,'Alleles'] = Final_SNP_Selection2['Alleles of Associated SNP'][x]
    Final_SNP_0_9.at[index2,'Mean Contribution of Main SNP'] = Final_SNP_Selection2['Mean Contribution of main SNP'][x]
    Final_SNP_0_9.at[index2,'Correlation'] = Final_SNP_Selection2['Correlation'][x]
    index2 += 1

Final_SNP_0_9 = Final_SNP_0_9.set_index('SNP Locus')
Final_SNP_0_9

### Interpreting the Model: Mean SHAP Value (contribution to each class)

In [ ]:
shap_values = shap.Explainer(clf_xgb1).shap_values(X_test1)
max_0 = np.sum(abs(shap_values).mean(0)[:,0] > 0 ) + 1
max_1 = np.sum(abs(shap_values).mean(0)[:,1] > 0 ) + 1 
max_2 = np.sum(abs(shap_values).mean(0)[:,2] > 0 ) + 1

labels = ["V/V", "V/M", "M/M"]
explainer = shap.Explainer(clf_xgb1)
shap_values = explainer(X_test1)

maxy = max(max_0,max_1,max_2) # obtain max number of SNPs amongst each class
max_x_abs = np.max(np.average(np.abs(shap_values.values), axis=(0))) # Obtain SNP importance of the  to set it as x axis limit

plt.subplot(1,3,1)
shap.plots.bar(shap_values[:,:,0], max_display= maxy, show=False)
plt.title(labels[0] + f' (n={cat_0})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='x-large')
plt.yticks(fontsize='x-large', fontweight='bold')
plt.xlabel('mean (|SHAP value|)', fontsize='x-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(0, max_x_abs)

plt.subplot(1,3,2)
shap.plots.bar(shap_values[:,:,1], max_display= maxy, show=False)
plt.title(labels[1] + f' (n={cat_1})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='x-large')
plt.yticks(fontsize='x-large', fontweight='bold')
plt.xlabel('mean (|SHAP value|)', fontsize='x-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(0, max_x_abs)

plt.subplot(1,3,3)
shap.plots.bar(shap_values[:,:,2], max_display= maxy, show=False)
plt.title(labels[2] + f' (n={cat_2})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='x-large')
plt.yticks(fontsize='x-large', fontweight='bold')
plt.xlabel('mean (|SHAP value|)', fontsize='x-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(0, max_x_abs)

plt.subplots_adjust(top=1, bottom=0.05, left=0.05, right=2.5, hspace=0.3, wspace=0.8)
filename = f'Intron {Intron} | V470M | Barplot.pdf'
file_path = os.path.join(directory_name, filename)
plt.savefig(file_path, format='pdf', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
min_x = np.min(shap_values.values) - (abs(np.min(shap_values.values))*0.01)  # obtain minimum shap value to set x-axis limit, giving slight buffer space aswell
max_x = np.max(shap_values.values) + (abs(np.max(shap_values.values))*0.01) # obtain maximum shap value to set x-axis limit, giving slight buffer space aswell 

plt.subplot(1,3,1)
shap.plots.beeswarm(shap_values[:,:,0], max_display= maxy, color_bar_label='Genotype Call Value of SNP',show=False)
cbar = plt.gcf().axes[-1]
vmin, vmax = cbar.get_ylim()
new_ticks = [vmin, (vmin+vmax)/2, vmax]
new_labels = ['0/0', '0/1\nor\n1/0', '1/1']
cbar.set_yticks(new_ticks)
cbar.set_yticklabels(new_labels)
cbar.yaxis.set_label_coords(13, 0.5)
plt.title(labels[0] + f' (n={cat_0})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='x-large')
plt.yticks(fontsize='x-large', fontweight='bold')
plt.xlabel('SHAP value', fontsize='x-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(min_x, max_x)

plt.subplot(1,3,2)
shap.plots.beeswarm(shap_values[:,:,1], max_display= maxy, color_bar_label='Genotype Call Value of SNP',show=False)
cbar = plt.gcf().axes[-1]
vmin, vmax = cbar.get_ylim()
new_ticks = [vmin, (vmin+vmax)/2, vmax]
new_labels = ['0/0', '0/1\nor\n1/0', '1/1']
cbar.set_yticks(new_ticks)
cbar.set_yticklabels(new_labels)
cbar.yaxis.set_label_coords(13, 0.5)
plt.title(labels[1] + f' (n={cat_1})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='x-large')
plt.yticks(fontsize='x-large', fontweight='bold')
plt.xlabel('SHAP value', fontsize='x-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(min_x, max_x)

plt.subplot(1,3,3)
shap.plots.beeswarm(shap_values[:,:,2], max_display= maxy, color_bar_label='Genotype Call Value of SNP',show=False)
cbar = plt.gcf().axes[-1]
vmin, vmax = cbar.get_ylim()
new_ticks = [vmin, (vmin+vmax)/2, vmax]
new_labels = ['0/0', '0/1\nor\n1/0', '1/1']
cbar.set_yticks(new_ticks)
cbar.set_yticklabels(new_labels)
cbar.yaxis.set_label_coords(13, 0.5)
plt.title(labels[2] + f' (n={cat_2})', fontweight='bold', size=16, color='black')
plt.xticks(fontsize='x-large')
plt.yticks(fontsize='x-large', fontweight='bold')
plt.xlabel('SHAP value', fontsize='x-large', fontweight='bold')
plt.locator_params(axis='x', nbins=5)
plt.xlim(min_x, max_x)

plt.subplots_adjust(top=1, bottom=0.05, left=0.05, right=2.5, hspace=0.3, wspace=0.8)
filename = f'Intron {Intron} | V470M | Dotplot.pdf'
file_path = os.path.join(directory_name, filename)
plt.savefig(file_path, format='pdf', dpi=600, bbox_inches='tight')
plt.show()

### Assessing Model Performance: Confusion Matrix

In [ ]:
# Building a confusion matrix 
cm1 = confusion_matrix(y_test1, y_pred1)
cm1_percent = cm1 / (cm1.sum(axis=1)[:, np.newaxis])

disp1 = ConfusionMatrixDisplay(confusion_matrix = cm1, display_labels= ["V/V", "V/M", "M/M"])
disp2 = ConfusionMatrixDisplay(confusion_matrix = cm1_percent, display_labels= ["V/V", "V/M", "M/M"])
fig, ax = plt.subplots(1,2, figsize=(24,12))
disp1.plot(ax=ax[0])
disp2.plot(ax=ax[1], values_format='.0%')
disp1.ax_.set_xticklabels(disp1.ax_.get_xticklabels(), rotation=45, ha='right', fontsize=24)
disp1.ax_.set_yticklabels(disp1.ax_.get_yticklabels(), fontsize=24)
disp1.ax_.set_ylabel('True Label', fontsize=24, fontweight='bold')
disp1.ax_.set_xlabel('Predicted Label', fontsize=24, fontweight='bold')
disp2.ax_.set_xticklabels(disp2.ax_.get_xticklabels(), rotation=45, ha='right', fontsize=24)
disp2.ax_.set_yticklabels(disp2.ax_.get_yticklabels(), fontsize=24)
disp2.ax_.set_ylabel('True Label', fontsize=24, fontweight='bold')
disp2.ax_.set_xlabel('Predicted Label', fontsize=24, fontweight='bold')
disp1.ax_.xaxis.labelpad = 16
disp2.ax_.xaxis.labelpad = 16
disp1.ax_.yaxis.labelpad = 16
disp2.ax_.yaxis.labelpad = 16

# Change font size within the confusion matrix
for axis in ax:
    for text in axis.texts:
        text.set_fontsize(26)
        
# Add red borders diagonally 
for i in range(len(cm1)):
    disp1.ax_.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1, fill=False, edgecolor='red', lw=3, zorder = 10))
for i in range(len(cm1_percent)):
    disp2.ax_.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1, fill=False, edgecolor='red', lw=3, zorder = 10))

# Color bar formatting
cbar1 = disp1.im_.colorbar
cbar1.ax.yaxis.label.set_size(22)
cbar1.ax.yaxis.set_tick_params(labelsize=22)

cbar2 = disp2.im_.colorbar
cbar2.ax.yaxis.label.set_size(22)
cbar2.ax.yaxis.set_tick_params(labelsize=22)


plt.subplots_adjust(wspace=0.42)
filename = f'Intron {Intron} | V470M | Confusion Matrix.pdf'
file_path = os.path.join(directory_name, filename)
plt.savefig(file_path, format='pdf', dpi=600, bbox_inches='tight')
fig.show()

### Assessing Model Performance: Precision-Recall Curves

In [ ]:
# Convert y_test1 to the same format as y_pred1 (rather than having an array of shape [77,1] have an array with shape [77,3] with each column depicting which class the individual is)
y_pred1 = clf_xgb1.predict_proba(X_test1)
y_pred1_multiclass = clf_xgb1.predict_proba(X_test1)
y_test1_temp = y_test1.reset_index(drop=True).copy()
y_test1_multiclass = np.zeros((y_pred1.shape[0],y_pred1.shape[1]))

for x in range(y_test1_temp.shape[0]):
    if y_test1_temp[x] == 0:
        y_test1_multiclass[x,0] = 1
        y_test1_multiclass[x,1] = 0
        y_test1_multiclass[x,2] = 0

    elif y_test1_temp[x] == 1:
        y_test1_multiclass[x,0] = 0
        y_test1_multiclass[x,1] = 1
        y_test1_multiclass[x,2] = 0

    elif y_test1_temp[x] == 2:
        y_test1_multiclass[x,0] = 0
        y_test1_multiclass[x,1] = 0
        y_test1_multiclass[x,2] = 1

In [ ]:
# Calculate Precision, Recall and Thresholds 
precision = dict()
recall = dict()
average_precision = dict()
for i in range(3):
    precision[i], recall[i], _ = precision_recall_curve(y_test1_multiclass[:, i], y_pred1_multiclass[:, i])
    average_precision[i] = average_precision_score(y_test1_multiclass[:, i], y_pred1_multiclass[:, i])

precision["micro"], recall["micro"], _ = precision_recall_curve(
    y_test1_multiclass.ravel(), y_pred1_multiclass.ravel()
)
average_precision["micro"] = average_precision_score(y_test1_multiclass, y_pred1_multiclass, average="micro")
average_precision['macro'] = (average_precision[0] + average_precision[1] + average_precision[2])/3

In [ ]:
# Visualize the Precision-Recall Curves, add f1 curves for reference 
labely = []
labely.append('V/V')
labely.append('V/M')
labely.append('M/M')
colors = cycle(['blue', 'orange', 'green'])

_, ax = plt.subplots(figsize=(11, 11))

f_scores = np.linspace(0.2, 0.8, num=4)
lines, labels = [], []
for f_score in f_scores:
    x = np.linspace(0.01, 1)
    y = f_score * x / (2 * x - f_score)
    (l,) = plt.plot(x[y >= 0], y[y >= 0], color="gray", alpha=0.2)
    plt.annotate("f1={0:0.1f}".format(f_score), xy=(0.9, y[45] + 0.02), fontsize = 15)

display = PrecisionRecallDisplay(
    recall=recall["micro"],
    precision=precision["micro"],
    average_precision=average_precision["micro"],
)
display.plot(ax=ax, name="Micro-average of all classes", color="gold")

for i, color in zip(range(3), colors):
    display = PrecisionRecallDisplay(
        recall=recall[i],
        precision=precision[i],
        average_precision= average_precision[i],
    )
    display.plot(ax=ax, name=f"{labely[i]}", color=color)

handles, labels = display.ax_.get_legend_handles_labels()
handles.extend([l])
labels.extend(["iso-f1 curves"])

ax.legend(handles=handles, labels=labels, loc="lower left", fontsize='xx-large')
ax.set_xlabel('Recall', fontsize='xx-large', fontweight='bold')
ax.set_ylabel('Precision', fontsize='xx-large', fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(),fontsize='xx-large')
ax.set_yticklabels(ax.get_yticklabels(),fontsize='xx-large')
custom_ticks = [0, 0.2, 0.4, 0.6, 0.8, 1]
ax.set_xticklabels(custom_ticks,fontsize='xx-large')
ax.set_yticklabels(custom_ticks,fontsize='xx-large')
plt.xlim(0, 1.005)
plt.ylim(0, 1.005)

filename = f'Intron {Intron} | V470M | PR Curve.pdf'
file_path = os.path.join(directory_name, filename)
plt.savefig(file_path, format='pdf', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# Tabulate the Average Precision of each class 
PRAUC_table = pd.DataFrame(columns= ['Macro Averaged AP' , 'Micro Averaged AP' , 'AP for V/V', 'AP for V/M', 'AP for M/M'], index= [f'Intron {Intron}'])

PRAUC_table.loc[f'Intron {Intron}', 'Micro Averaged AP' ] = average_precision['micro']
PRAUC_table.loc[f'Intron {Intron}', 'AP for V/V'] = average_precision[0]
PRAUC_table.loc[f'Intron {Intron}', 'AP for V/M'] = average_precision[1]
PRAUC_table.loc[f'Intron {Intron}', 'AP for M/M'] = average_precision[2]
#PRAUC_table.loc[f'Intron {Intron}', 'Macro Averaged 5 fold CV AP of Training Set'] = round(grid_search.best_score_,3)
PRAUC_table.loc[f'Intron {Intron}', 'Macro Averaged AP' ] = average_precision['macro']

PRAUC_table = PRAUC_table
#PRAUC_table.to_excel(f'Intron {Intron} | AP | V470M.xlsx', index=True)
PRAUC_table.to_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {Intron} | AP | V470M.xlsx', index=True)

### Perform a statistical evaluation of Intronic SNP Dosage & V470M  Dosage association

In [ ]:
# 0.7 Correlation Statistical Evaluation 
important_columns = np.abs(shap_values.values).mean(axis=(0,2)) # calculate mean for each SNP 
important_columns = important_columns > 0 # get all SNPs with a mean SHAP above 0
important_columns_names2 = X_test1.columns[important_columns] # get names for each SNP deemed important by the model (not including associated SNPs)
important_columns_names = Final_SNP_0_7.index # get names for each SNP deemed important by the model (including associated SNPs)

calculated_dosage = pd.DataFrame(index= (important_columns_names), columns=('SNP dosage associated with V/V Polymorphism', 'Odds-Ratio (V/V)', 'P-Value (V/V)', 'Significant? (V/V)',
                                                                           'SNP dosage associated with V/M Polymorphism', 'Odds-Ratio (V/M)', 'P-Value (V/M)', 'Significant? (V/M)',
                                                                           'SNP dosage associated with M/M Polymorphism', 'Odds-Ratio (M/M)', 'P-Value (M/M)', 'Significant? (M/M)'))

genotype_order_1 = ['SNP dosage associated with V/V Polymorphism','SNP dosage associated with V/M Polymorphism','SNP dosage associated with M/M Polymorphism']
genotype_order_2 = ['Odds-Ratio (V/V)', 'Odds-Ratio (V/M)', 'Odds-Ratio (M/M)']
genotype_order_3 = ['P-Value (V/V)', 'P-Value (V/M)', 'P-Value (M/M)']
genotype_order_4 = ['Significant? (V/V)', 'Significant? (V/M)', 'Significant? (M/M)']

for feature in important_columns_names:
    feature_data = shap_values

    no_poly= X_test1[feature] == 0 # all individuals with V/V
    heterozygous = X_test1[feature] == 0.5 # all individuals with V/M
    homozygous = X_test1[feature] == 1 # all individuals with M/M

    for genotype in range(shap_values.values.shape[2]):

        # If this SNP is associated to a more significant SNP (per the model) assign the dosages as determined by the model for that SNP to this SNP 
        if feature not in important_columns_names2:
            associated_snp = Final_SNP_0_7.loc[feature, 'Associated SNP Locus']
            highest_shap_genotype = calculated_dosage.loc[associated_snp,genotype_order_1[genotype]]

        else:
            # For each genotype, calculate the shap value for each dosage 
            if (shap_values.values[no_poly,X_test1.columns.get_loc(feature), genotype]).size != 0:
                no_poly_shap = np.mean(shap_values.values[no_poly,X_test1.columns.get_loc(feature), genotype])
            else: 
                no_poly_shap = np.nan
            if (shap_values.values[heterozygous,X_test1.columns.get_loc(feature), genotype]).size != 0:
                heterozygous_shap = np.mean(shap_values.values[heterozygous,X_test1.columns.get_loc(feature), genotype])
            else:
                heterozygous_shap = np.nan
            if (shap_values.values[homozygous,X_test1.columns.get_loc(feature), genotype]).size != 0:
                homozygous_shap = np.mean(shap_values.values[homozygous,X_test1.columns.get_loc(feature), genotype])
            else: 
                homozygous_shap = np.nan
            
            # create a key to determine which dosage has the highest mean shap
            shap_dict = {"No Polymorphism": no_poly_shap, "Heterozygous Polymorphism": heterozygous_shap, "Homozygous Polymorphism": homozygous_shap} # order such that equal average mean shaps will yield None as the highest shap genotype 
            highest_shap_genotype = max(shap_dict, key=shap_dict.get) 
            if ((shap_dict[highest_shap_genotype] == 0) or (np.isnan(shap_dict[highest_shap_genotype]))):
                highest_shap_genotype = 'No Prediction'
        
        calculated_dosage.loc[feature, genotype_order_1[genotype]] = highest_shap_genotype
        if highest_shap_genotype == 'No Prediction':
            continue

        # Calculate p-value 
        combined = temp_data[[feature,'V470M']]
        combined_vals = combined.value_counts() # get count of values 
        
        # Perform Fisher's exact test (greater) tp statistically determine the significance of the association findings 
        if highest_shap_genotype == "No Polymorphism":
            a = combined_vals[(combined_vals.index.get_level_values(0) == 0) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is 0 and where there is V470M Polymorphism of interest
            b = combined_vals[(combined_vals.index.get_level_values(0) == 0) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is 0 and no V470M Polymorphism of interest
            c = combined_vals[(combined_vals.index.get_level_values(0) != 0) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is anything but 0 and there is V470M Polymorphism of interest
            d = combined_vals[(combined_vals.index.get_level_values(0) != 0) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is anything but 0 and no V470M Polymorphism of interest

        elif highest_shap_genotype == "Heterozygous Polymorphism":
            a = combined_vals[(combined_vals.index.get_level_values(0) == 0.5) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is 0.5 and where there is V470M Polymorphism of interest
            b = combined_vals[(combined_vals.index.get_level_values(0) == 0.5) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is 0.5 and no V470M Polymorphism of interest
            c = combined_vals[(combined_vals.index.get_level_values(0) != 0.5) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is anything but 0.5 and there is V470M Polymorphism of interest
            d = combined_vals[(combined_vals.index.get_level_values(0) != 0.5) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is anything but 0.5 and no V470M Polymorphism of interest
         
        elif highest_shap_genotype == "Homozygous Polymorphism":
            a = combined_vals[(combined_vals.index.get_level_values(0) == 1) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is 1 and where there is V470M Polymorphism of interest
            b = combined_vals[(combined_vals.index.get_level_values(0) == 1) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is 1 and no V470M Polymorphism of interest
            c = combined_vals[(combined_vals.index.get_level_values(0) != 1) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is anything but 1 and there is V470M Polymorphism of interest
            d = combined_vals[(combined_vals.index.get_level_values(0) != 1) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is anything but 1 and no V470M Polymorphism of interest

    
        table = np.array([[a,b], [c,d]])
        odds_ratio, p_value = fisher_exact(table,alternative='greater')
        if np.isinf(odds_ratio) or np.isnan(odds_ratio): # inf if b or c == 0 , nan if a or d = 0 while b or c = 0 
            if ((a == 0 and b == 0) or (c == 0 and d == 0) or (a == 0 and c == 0) or (b == 0 and d == 0)):
                odds_ratio = 'Undefined (Unknown Direction)'
            elif (b == 0 or c == 0):
                odds_ratio = '∞ (Strong Positive Association)'
        else:
            odds_ratio = float(f"{odds_ratio:.2f}")

        calculated_dosage.loc[feature, genotype_order_2[genotype]] = odds_ratio
        calculated_dosage.loc[feature, genotype_order_3[genotype]] = f"{p_value:.2e}" if p_value < 0.001 else f"{p_value:.3f}"

        if isinstance(odds_ratio, str):
            if ((odds_ratio == '∞ (Strong Positive Association)') and (float(p_value)< 0.05)):
                calculated_dosage.loc[feature, genotype_order_4[genotype]] = 'yes'
            else: 
                calculated_dosage.loc[feature, genotype_order_4[genotype]] = 'no'
        else: 
            if ((odds_ratio > 1) and (float(p_value) < 0.05)):
                calculated_dosage.loc[feature, genotype_order_4[genotype]] = 'yes'
            else: 
                calculated_dosage.loc[feature, genotype_order_4[genotype]] = 'no'

calculated_dosage.replace(np.nan, '', inplace=True)
Final_SNP_0_7 = Final_SNP_0_7.join(calculated_dosage)
Final_SNP_0_7['main SNP'] = Final_SNP_0_7.apply(lambda x: True if x['Associated SNP Locus'] == '' else False, axis = 1)
Final_SNP_0_7 = Final_SNP_0_7.sort_values(by=['Mean Contribution of Main SNP', 'main SNP', 'Correlation'], ascending = [False, False, False])
Final_SNP_0_7 = Final_SNP_0_7.drop(columns='main SNP')
Final_SNP_0_7.to_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {Intron} | Final SNPs (0.7 Correlation Threshold) | V470M.xlsx', index=True)
Final_SNP_0_7